# Start

# Query clauses

SQL query clauses:
- `SELECT`
- `FROM`
- `WHERE`
- `GROUP BY`
- `HAVING`
- `ORDER BY`

Each clause has keywords / statements, e.g. clause SELECT has statements such as DISTINCT etc.

**Order of execution**:

1. FROM / JOIN
2. WHERE
3. GROUP BY
4. HAVING
5. SELECT
6. DISTINCT
7. ORDER BY 
8. LIMIT
9. OFFSET

**Aliases**

```sql
-- General syntax
SELECT 
  column1 AS "column title", 
  column2 AS aliasHere, ..., columnN
-- or `*` to select the rows from all the columns in a table
-- or `table1.column1, table1.column2` to specify which table, especially useful in joins
FROM table1
WHERE 
  column2 = 'Value' -- allows us to specify a condition by using an operator
  AND (column3 = 'Value2' OR column3 > 100);

-- we can also give aliases to the tables
SELECT o.OrderId, o.OrderDate, c.CustomerId, c.FirstName, c.LastName, c.Country
FROM Orders o
RIGHT JOIN Customers c 
ON o.CustomerId = c.CustomerId

-- When aliasing, we can either use the keyword AS or omit it
-- (albeit it is preferable to use the AS keyword)
SELECT column1 AS alias1 
-- or
SELECT column1 alias1
```

> Note: alias name cannot start with a digit / number

## SELECT

```sql
-- select all columns
SELECT * 
FROM table1
-- select all but one column
SELECT * EXCEPT (id)
FROM table1
```

There are different ways of aliasing columns:

```sql
SELECT
  column1, 
  COUNT(*)
FROM table1
GROUP BY column1
ORDER BY COUNT(*) DESC

-- or
SELECT
  column1, 
  COUNT(*)
FROM table1
GROUP BY 1
ORDER BY 2 DESC
```

Built-in functions:

```sql
-- you don't even need a WHERE clause
SELECT
    version(),
    user(),
    database()
;
```

### COUNT

### Record expansion

In PostgreSQL, if you have a column where each row is a record/tuple of fixed length (e.g., `(a, b, c)`), and you want to split this column into separate columns, you can use record expansion (also called row deconstruction) via:
- `SELECT (column_name).*` syntax
- Or explicitly cast to a named composite type or use `unnest`, `json`, etc., depending on the context

For instance, the query below returns a column which consists of records with tuples of fixed length:
```sql
WITH temp1 AS (
	SELECT 1 a, 2 b, 3 c 
	UNION ALL
	SELECT 4 a, 5 b, 6 c
), 
temp2 AS (
	SELECT DISTINCT(a, b, c)
	FROM temp1
)
SELECT *
FROM temp2 
-- row    |
-- -------+
-- (1,2,3)|
-- (4,5,6)|
```

You can split it into several columns 
```sql
WITH temp1 AS (
	SELECT 1 a, 2 b, 3 c 
	UNION ALL
	SELECT 4 a, 5 b, 6 c
), 
temp2 AS (
	SELECT DISTINCT(a, b, c)
	FROM temp1
)
SELECT 
	(row).f1 AS row1,
	(row).f2 AS row2,
	(row).f3 AS row3
FROM temp2 
-- row1|row2|row3|
-- ----+----+----+
--    1|   2|   3|
--    4|   5|   6|
```

### EXCEPT

This function doesn't work in PostgreSQL. Works in BigQuery.

```sql
-- Select every column except for the column `alias`
SELECT
	* EXCEPT (alias)
FROM student s

```

### DISTINCT

```sql
-- Only print unique values from the column
-- the two queries below produce the same result
SELECT DISTINCT (column1)
SELECT DISTINCT column1

-- Show unique combinations of two columns - returns column where each row is a list of unique combs
SELECT DISTINCT (column1, column2)
-- show unique combinations of two columns BUT return a normal table
SELECT DISTINCT column1, column2

-- example
SELECT DISTINCT(country, lang)
FROM (
	SELECT 'usa' AS country, 'english' AS lang
	UNION ALL
	SELECT 'usa' AS country, 'spanish' AS lang
	UNION ALL
	SELECT 'can' AS country, 'french' AS lang
	UNION ALL
	SELECT 'usa' AS country, 'spanish' AS lang
) AS subquery1;
-- `SELECT DISTINCT(country, lang)` returns this:
-- row          |
-- -------------+
-- (can,french) |
-- (usa,english)|
-- (usa,spanish)|

-- `SELECT DISTINCT country, lang` returns this:
-- country|lang   |
-- -------+-------+
-- usa    |spanish|
-- can    |french |
-- usa    |english|
```


In [ ]:
SELECT
    COUNT(customer_id) AS num_rows,
    COUNT(DISTINCT customer_id) AS num_unique_customers
FROM table1;

### COALESCE

> Basically null-value handling.
>
> NOTE: the order of arguments in COALESCE is very important - it returns the FIRST non-null value, therefore, you can thus set priority - the first argument has a higher priority than the second argument, where the latter is used only if the former is absent.

Return the first non-null value in a list of columns. If all the values in the list of columns are NULL, then the function returns NULL

```sql
SELECT 
	name 
	, alias 
	, COALESCE(name, alias)
FROM student s 
-- returns:
-- name        |alias       |coalesce    |
-- ------------+------------+------------+
-- John Wick   |Baba Yaga   |John Wick   |
-- Jack Bauer  |Dude from 24|Jack Bauer  |
-- Poseidon    |Tlalok      |Poseidon    |
-- John Stramer|            |John Stramer|
--             |Ghostface   |Ghostface   |


-- in this case get a value for NULL values
WITH table1 AS (
	SELECT 'John' AS name
	UNION ALL
	SELECT NULL AS name
	UNION ALL
	SELECT 'Jack' AS name
)
SELECT 
	COALESCE(name, 'unknown')
FROM table1
-- coalesce|
-- --------+
-- John    |
-- unknown |
-- Jack    |
```

Can also coalesce based on three columns:
```sql
WITH table1 AS (
	SELECT 'John' AS name1, 'John' AS name2, NULL AS name3
	UNION ALL
	SELECT NULL AS name1, NULL AS name2, 'Jane' AS name3
	UNION ALL
	SELECT 'Jack' AS name1, 'Jill' AS name2, 'Joan' AS name3
	UNION ALL
	SELECT NULL AS name1, NULL AS name2, NULL AS name3
)
SELECT 
	COALESCE(name1, name2, name3),
	*
FROM table1

-- coalesce|name1|name2|name3|
-- --------+-----+-----+-----+
-- John    |John |John |     |
-- Jane    |     |     |Jane |
-- Jack    |Jack |Jill |Joan |
--         |     |     |     |
```

You can also include COALESCE into a WHERE condition:
```sql
with temp1 as (
  select 'ab' id 
  union all
  select 'cd' id 
  union all 
  select NULL id
)
select 
  coalesce(id, 'none') AS id
from temp1
where coalesce(id, 'none') <> 'ab'
```

### Row array-like, SPLIT

Splits a string row into multiple rows based on a specified delimiter. 

> Works in BigQuery
>
> Doesn't work in PostgreSQL

By default, splits the rows into nested rows, where each row will be a list of split values:

```sql
WITH temp1 AS (
  SELECT 0 id, SPLIT('product a', ' ; ') feature
  UNION ALL
  SELECT 1 id, SPLIT('product a ; product b ; product c', ' ; ') feature
  UNION ALL 
  SELECT 2 id, SPLIT('product a ; product b', ' ; ') feature
)
SELECT *
FROM temp1

-- [{
--   "id": "0",
--   "feature_split": ["product a"]
-- }, {
--   "id": "1",
--   "feature_split": ["product a", "product b", "product c"]
-- }, {
--   "id": "2",
--   "feature_split": ["product a", "product b"]
-- }]

-- As a table in database IDE, it kind of looks like this:
-- | id | feature   |
-- | -- | --------- |
-- | 0  | product a |
-- | 1  | product a |
-- |    | product b |
-- |    | product c |
-- | 2  | product a |
-- |    | product b |
```

To reset the index, use the UNNEST function:
```sql
WITH temp1 AS (
  SELECT 0 id, SPLIT('product a', ' ; ') feature
  UNION ALL
  SELECT 1 id, SPLIT('product a ; product b ; product c', ' ; ') feature
  UNION ALL 
  SELECT 2 id, SPLIT('product a ; product b', ' ; ') feature
)
SELECT 
  id,
  feature_split_2
FROM 
  temp1,
  UNNEST(feature) AS feature_split_2

-- [{
--   "id": "0",
--   "feature_split_2": "product a"
-- }, {
--   "id": "1",
--   "feature_split_2": "product a"
-- }, {
--   "id": "1",
--   "feature_split_2": "product b"
-- }, {
--   "id": "1",
--   "feature_split_2": "product c"
-- }, {
--   "id": "2",
--   "feature_split_2": "product a"
-- }, {
--   "id": "2",
--   "feature_split_2": "product b"
-- }]
```

Now check this out! An alternative to `SPLIT_PART` in BigQuery is the combination of `SPLIT` and `SAFE_OFFSET`:
```sql

with temp1 AS (
  select 'a/b/c' col1
  union all
  select 'a2/b2' col1 
  union all 
  select 'a3' col1
)

select 
  col1, 
  split(col1, '/')[safe_offset(0)]
from temp1
```


At least in BigQuery, you can filter based on these array-like values:
```sql
WITH temp1 AS (
  SELECT 0 id, split('product a', ' ; ') feature
  UNION ALL
  SELECT 1 id, split('product a ; product b ; product c', ' ; ') feature
  UNION ALL 
  SELECT 2 id, split('product a ; product b', ' ; ') feature
)

SELECT *
FROM temp1
WHERE array_to_string(feature, ',') = 'product a,product b'
-- [{
--   "id": "2",
--   "feature": ["product a", "product b"]
-- }]
```


## FROM

> The **FROM** clause defines the tables used by a query, along with the means of linking the tables together

To see which types of tables you can use in your FROM clause, see the `Types of tables` section.

## WHERE


The WHERE clause is used in a SELECT statement to filter rows based on the specified *filter conditions / statements* before the data is grouped or aggregated. It operates on individual rows and filters them based on the given conditions.

- WHERE clause can have multiple filter conditions separated by the operators `AND` or `OR`

`WHERE salary IS NOT NULL`

```sql
-- OR condition
WHERE 
  column1 != 2 
  OR column2 IS NULL
--
WHERE 
  (surname = 'Jones' AND age > 17)
  OR (surname = 'Wilkinson' AND age > 50)



### EXISTS

The `EXISTS` operator is used to test for the existence of any record in a subquery; returns TRUE if the subquery returns one or more records. 

Examples:

The following SQL statement returns TRUE and lists the suppliers with a product price less than 20:

```sql
WITH Products AS (
  SELECT 1 ProductID, 'Chais' ProductName, 1 SupplierID, 1 CategoryID, '10 boxes x 20 bags' Unit, 18 Price union all 
  select 2, 'Chang', 1, 1, '24 - 12 oz bottles', 19 union all 
  select 3, 'Aniseed Syrup', 1, 2, '12 - 550 ml bottles', 10 union all
  select 4, "Chef Anton's Cajun Seasoning", 2, 2, "48 - 6 oz jars", 22 union all 
  select 5, "Chef Anton's Gumbo Mix", 2, 2, "36 boxes", 21.35
),

Suppliers AS (
  select 1 SupplierId, 'Exotic Liquid' SupplierName, 'Charlotte Cooper' ContactName union all 
  select 2, 'New Orleans Cajun Delights', 'Shelley Burke'
)

-- Select suppliers with a product price less than 20
SELECT SupplierName
FROM Suppliers 
WHERE EXISTS (
  SELECT ProductName
  FROM Products 
  WHERE
    Products.SupplierID = Suppliers.supplierID
    AND Price < 20
);
-- [{
--   "SupplierName": "Exotic Liquid"
-- }]
```


### BETWEEN

In [ ]:
-- BETWEEN (PostgreSQL, MySQL)
-- Inclusive between <lower limit> and <upper limit>; >= lower_limit AND <= upper_limit
age BETWEEN 25 AND 30
-- Values between two dates
date BETWEEN '1999-01-01' AND '2015-01-01' -- between 1991-01-01 00:00:00 (midnight) and 2015-01-01 00:00:00 (midnight)
-- Values alphabetically between two strings
column1 BETWEEN 'Alpha' AND 'Beta'
-- Include names like 'FARNELL', 'FENNEL', 'FRANKLIN', 'FRAZIER'
column1 BETWEEN 'FA' and 'FRB' -- if you put 'FR' instead of 'FRB', it won't include 'FRANKLIN' and 'FRAZIER'

## GROUP BY (explicit groups)

Can be:
- Single-column grouping: `GROUP BY column1`
- Multicolumn grouping: `GROUP BY column1, column2`
- Grouping via expressions: `GROUP BY EXTRACT(YEAR FROM rundate)`

> Note: you can either use WHERE or HAVING in GROUP BY for filtering. Please see the respective section for the differences.


> If you have multiple columns in your ORDER BY clause, the order in which columns appear there matter, as one row might appear before the other if you order by multiple columns in different order.

```sql
-- General form
SELECT column1, column2, ..., columnN
FROM table_name
ORDER BY 
  column1 [ASC|DESC], 
  column2 [ASC|DESC], 
  ... 
  columnN [ASC|DESC];

-- Examples
SELECT *
FROM employees
ORDER BY salary DESC, age DESC;

-- ORDER BY always comes after GROUP BY
SELECT 
  country, 
  COUNT(*) AS n_companies
FROM companies
GROUP BY country
ORDER BY n_companies DESC
LIMIT 10
```

You can also sort the columns using the **numeric placeholders**:
- It's useful when you are sorting on expressions
```sql
SELECT 
  name,
  surname,
  age,
  birthday
FROM person
ORDER BY 3 DESC; -- order the table using the third element in the SELECT clause
```

Please note that sort is case-sensitive, i.e. SORT BY will sort `amazon, Avon, Bangladesh` to `Avon, Bangladesh, amazon`, so all lowercase will be ordered after all uppercase. To prevent this, do this:
```sql
SELECT 
  col1, 
  col2
FROM temp1
ORDER BY LOWER(col1)
```


```sql
-- GROUP BY
--- we can use order of the tables in the filter statement - you basically substitute the names of columns in the filter statement with their ordinal number (index in the order of mention)
SELECT 
  column1, 
  column2, 
  COUNT(column3)
FROM table1
GROUP BY 1 2 
ORDER BY 2 DESC

-- Find out the total salary paid out by each department
SELECT 
  department_id, 
  SUM(salary) as total_salary
FROM employees
GROUP BY department_id;
-- Group by can be used with joins:
SELECT u.name as NAME, SUM(t.amount) as BALANCE
FROM Users u
INNER JOIN Transactions t
ON u.account = t.account
GROUP BY u.name
HAVING SUM(t.amount) > 10000


-- Note: order of GROUP BY doesn't matter - the final numbers will remain the sum, just the order will change
-- https://www.kaggle.com/discussions/getting-started/100307
-- https://stackoverflow.com/questions/3064677/does-the-order-of-columns-matter-in-a-group-by-clause
-- consider this table:
-- | country | person | sale |
-- | - | - | - |
-- | UK | John | 100 |
-- | UK | John | 200 |
-- | UK | Lisa | 500 |
-- | Mex | John | 100 |
-- | Mex | Marvin | 150 |
-- | Mex | Marvin | 150 |
-- | Mex | Jake | 50 |
-- Now consider two queries: 
SELECT 1, 2, SUM(sale)
FROM table1
GROUP BY 1, 2
-- or
SELECT 2, 1, SUM(sale)
FROM table1
GROUP BY 2, 1
-- they will produce the same calculations, just in different order
```

```sql
-- Count number of repetitions of unique categories in column `column1`
SELECT branch_id, COUNT(*) FROM branch_supplier GROUP BY branch_id 
-- Count number of repetitions of unique categories in column `column1` where count is greater than 3
SELECT branch_id, COUNT(*) FROM branch_supplier GROUP BY branch_id HAVING COUNT(*) > 3;

-- Count how many unique categories each super_id has
SELECT super_id, COUNT(DISTINCT(emp_id)) FROM employee GROUP BY super_id;
```

- `GROUP BY column1`
- `GROUP BY column1 HAVING COUNT(*) > 5` only group those values whose count is > 5
- `select major_id, count(*) from students group by major_id;` count unique values in column 'major_id'
- `select major_id, min(gpa) from students group by major_id;` view min value in each group within column major_id

### ALL

The `GROUP BY ALL` clause is a SQL shorthand notation that automatically groups the results by all non-aggregated columns present in the SELECT statement. This feature is a non-standard extension supported by some modern data platforms like Snowflake, Databricks, **BigQuery**, and ClickHouse, but not universally available in traditional relational databases like MySQL or Oracle. 

Example:
```sql
with temp1 AS (
  select 1 id, 'meat' category, 15 price union all 
  select 2 id, 'meat' category, 6 price union all 
  select 3 id, 'veggies' category, 10 price union all 
  select 4 id, 'veggies' category, 2 price union all 
  select 5 id, 'other' category, 10 price  
)

select
  category,
  sum(price)
FROM temp1 
group by all

-- output:
-- [{
--   "category": "meat",
--   "f0_": "21"
-- }, {
--   "category": "veggies",
--   "f0_": "12"
-- }, {
--   "category": "other",
--   "f0_": "10"
-- }]
```


### ROLLUP

Using ROLLUP also shows total counts (shown as NULL) for each subgroup. Can be used in single-column and multicolumn grouping.

- MySQL: `GROUP BY column1, column2 WITH ROLLUP`
- PostgreSQL: `GROUP BY ROLLUP (column_name1, column_name2)`

In [ ]:
/* Single-column grouping:
includes totals count per actor_id (shown as null classes).
*/
WITH temp1 AS (
	SELECT 1 actor_id, 1 movie_id UNION ALL 
	SELECT 1, 2 UNION ALL 
	SELECT 1, 3 UNION ALL 
	
	SELECT 2, 1 UNION ALL 
	
	SELECT 3, 3 UNION ALL 
	SELECT 3, 4
)

SELECT 
	actor_id,
	COUNT(movie_id)
FROM temp1
GROUP BY ROLLUP (1)
ORDER BY 1 ASC

-- actor_id|count|
-- --------+-----+
--        1|    3|
--        2|    1|
--        3|    2|
--         |    6|

In [ ]:
/* Multicolumn grouping:
includes totals count per actor_id and genre (shown as null classes)
*/
WITH temp1 AS (
	SELECT 1 actor_id, 1 genre, 1 movie_id UNION ALL 
	SELECT 1, 1, 2 UNION ALL 
	SELECT 1, 1, 3 UNION ALL 
	
	SELECT 2, 2, 1 UNION ALL 
	
	SELECT 3, 1, 3 UNION ALL 
	SELECT 3, 2, 4 UNION ALL 
	SELECT 3, 3, 5
)

SELECT 
	actor_id,
	genre,
	COUNT(movie_id)
FROM temp1
GROUP BY ROLLUP (1, 2)
ORDER BY 1 ASC, 2

-- actor_id|genre|count|
-- --------+-----+-----+
--        1|    1|    3|
--        1|     |    3|
--        2|    2|    1|
--        2|     |    1|
--        3|    1|    1|
--        3|    2|    1|
--        3|    3|    1|
--        3|     |    3|
--         |     |    7|

### WHERE vs HAVING

**WHERE vs HAVING clause**:
- WHERE is used for filtering rows **BEFORE** any grouping or aggregation. 
  - You cannot filter in your WHERE statements based on aggregate functions, as they haven't been generated yet. Therefore, WHERE does not work with aggregated results;
- HAVING is used for filtering rows **AFTER** any grouping or aggregation.
  - The HAVING clause was added to SQL to filter the results of the GROUP BY clause. 
  - The HAVING clause is used in combination with the GROUP BY clause in a SELECT statement to filter rows based on specified conditions after the data is grouped and aggregated. It operates on the result of the grouping operation and filters the aggregated data.

**Here is a thorough example:**
```sql
SELECT * 
FROM client;
-- client_id|branch_id|
-- ---------+---------+
--       400|        2|
--       401|        2|
--       402|        3|
--       403|        3|
--       404|        2|
--       405|        3|
--       406|        2|

/*
For example, in the query below, you can query BEFORE grouping, 
but you cannot query aggregate functions. For instance, this is a simple query condition
*/
WITH client AS (
	SELECT 400 client_id, 2 branch_id UNION ALL SELECT 401 client_id, 2 branch_id UNION ALL SELECT 402 client_id, 3 branch_id UNION ALL SELECT 403 client_id, 3 branch_id UNION ALL SELECT 404 client_id, 2 branch_id UNION ALL SELECT 405 client_id, 3 branch_id UNION ALL SELECT 406 client_id, 2 branch_id
)
SELECT 
	branch_id,
	COUNT(*) AS clients_per_branch
FROM client
WHERE client_id <> 405 -- however, you CANNOT write `WHERE COUNT(*) = 4`
GROUP BY branch_id;
-- branch_id|clients_per_branch|
-- ---------+------------------+
--         3|                 2|
--         2|                 4|

/*
If you wanted, however, to filter by the results of the count aggregate function, 
you would have to include an extra CTE, as you cannot filter using WHERE keyword 
the result of an aggregate function:
*/
WITH client AS (
	SELECT 400 client_id, 2 branch_id UNION ALL SELECT 401 client_id, 2 branch_id UNION ALL SELECT 402 client_id, 3 branch_id UNION ALL SELECT 403 client_id, 3 branch_id UNION ALL SELECT 404 client_id, 2 branch_id UNION ALL SELECT 405 client_id, 3 branch_id UNION ALL SELECT 406 client_id, 2 branch_id
), 
temp1 AS (
	SELECT 
	branch_id,
	COUNT(*) AS clients_per_branch
	FROM client
	WHERE client_id <> 405
	GROUP BY branch_id
)
SELECT 
	*
FROM temp1
WHERE clients_per_branch = 4
-- branch_id|clients_per_branch|
-- ---------+------------------+
--         2|                 4|

/*
Nevertheless, you can use HAVING statement to filter the result
of an aggregate function like COUNT:
*/
WITH client AS (
	SELECT 400 client_id, 2 branch_id UNION ALL SELECT 401 client_id, 2 branch_id UNION ALL SELECT 402 client_id, 3 branch_id UNION ALL SELECT 403 client_id, 3 branch_id UNION ALL SELECT 404 client_id, 2 branch_id UNION ALL SELECT 405 client_id, 3 branch_id UNION ALL SELECT 406 client_id, 2 branch_id
)
SELECT 
	branch_id,
	COUNT(*) AS clients_per_branch
FROM client 
GROUP BY branch_id
HAVING COUNT(*) = 4;
-- branch_id|clients_per_branch|
-- ---------+------------------+
--         2|                 4|
```

If you have both a WHERE clause and a HAVING clause in your query, WHERE will execute first.

In order to use HAVING, you also need:
- A GROUP BY clause
- An aggregation in your SELECT section (SUM, MIN, MAX, etc.)

Some more examples of using the `HAVING` statement:
```sql
-- Example 1
SELECT 
  p.name,
  p.surname,
  COUNT(*)
FROM person p
INNER JOIN transactions t
ON p.id = t.person_id
GROUP BY 
  name, 
  surname
HAVING COUNT(*) >= 40


-- Example 2
SELECT column1, aggregate_function(column2)
FROM table
GROUP BY column1
HAVING aggregated_condition;
-- Find out which departments have a total salary payout greater than 50,000
SELECT 
  department_id,
  COUNT(*) AS number_of_employees, -- also can be `COUNT(employee_id) AS number_of_employees` 
  SUM(salary) as total_salary
FROM employees
GROUP BY department_id
HAVING SUM(salary) > 50000;
```


## LIMIT

Show $n$ first rows.

## OFFSET

Skip $n$ rows.

> Note: for some reason, you can use OFFSET only after a LIMIT, but without LIMIT you can't use it

```sql
WITH temp1 AS (
  SELECT 0 index, 'A' letter
  UNION ALL
  SELECT 1 index, 'B' letter
  UNION ALL
  SELECT 2 index, 'C' letter
)
SELECT *
FROM temp1
LIMIT 10
OFFSET 2
```

# Data types

## Character

| Datatype | Description | Example |
| - | - | - |
| `CHAR(30)` | Fixed-length, blank-padded strings. <br> The string has to be EXACTLY the specified length, in this case, 30 characters - no more, no less. These are right-padded with spaces (to fill up the remaining characters not used by definition of a variable) and always consume the same number of bytes.| State abbreviations - all strings stored in the column are of the same length. |
| `VARCHAR(30)` | Variable-length string. The string can have a length up to the specified limit, such as 10, 20, 25 characters, but no more than e.g. 30 characters. | Varchar is appropriate for free-form data entry, e.g. notes column to hold data about customer interactions with your company's customer service department.  |
| PostgreSQL `text` ; MySQL `tinytext`, `text`, `mediumtext`, `longtext` | To store longer strings such as emails, XML documents. | Note: in MySQL, `tinytext` and `text` aren't normally used. Instead, you can see more often the use of mediumtext and longtext that can be used for storing documents. |

> Note 1: 
> for character data types, use single quotes, not doublequotes. 
> If you need to use a single apostrophe as part of the string, use it two times to escape, e.g. to write a string `O'Brien` you can escape like this: `'O''Brien'`
>
> HOWEVER, in BigQuery, you can use single quotes `'` or double quotes `"`, to represent character data types and datetime. 

> Note 2:
> CHAR and VARCHAR are for storing relatively short text strings. For longer, use text data types

Define character set:
```sql
-- for a variable
VARCHAR(20) CHARACTER SET latin1
-- for the entire database
CREATE DATABASE database1 CHARACTER SET latin1;
```

## Numeric

| Datatype | Description | Example |
| - | - | - |
| `SERIAL` | Auto-increments a number upon inserting a new row. The SERIAL type will make your column an INT with a NOT NULL constraint, and automatically increment the integer when a new row is added. `BIGSERIAL` is the same but has a higher range of possible values.  |
| `BOOLEAN` | `TRUE`, `FALSE` | A column indicating whether a customer order has been shipped |
| `INT` | Whole number. MySQL also has `tinyint`, `smallint`, `mediumint`, `int`, `bigint` | |
| `FLOAT` | Can be `FLOAT(p,s)`, where p is the total number of digits and s is number of allowable digits to the right of the decimal point. For MySQL, can be `FLOAT`, `FLOAT(p,s)`, and for even larger numbers `DOUBLE(p,s)`. | E.g. `FLOAT(4,2)` - handles 17.87, 8.19, but rounds 17.8675 to 17.87 and errors at attempt of storing 178.375 |

> Note 1: 
> The numeric data types can be defined as `unsigned`, meaning that they are greater than or equal to zero.

Some useful functions for doing math in SQL:
```sql
-- Power of a number
-- Works in PostgreSQL, MySQL, BigQuery
POW(base, exponent)
POWER(2, 3) -- 2^3
```

```sql
-- | `POW(2, 3)` | Power; in this example, 2^3. Works in PostgreSQL, MySQL. |
-- | `MOD(<number_to_round/column>, <number-by-which-to-divide>)` | Modulo: check the remainder of the division. In this case, remainder is zero if the number is even. E.g. `MOD(3, 2)`, `MOD(column1, 2)`. Works in MySQL and PostgreSQL. |
-- | `exp(x)` | Calculate the e^x |
-- | `ln(x)` | Calculate the natural log of x |
-- | `sqrt(x)` | Calculate the square root of x |

-- In a column 'comparison' with binary values (0 and 1), calculate percentage that all ones make from the total amount
SELECT ROUND( (SUM(comparison)::numeric / COUNT(comparison)::numeric) * 100 , 2 ) AS immediate_percentage

-- SIGN
-- Show the sign of the signed number
-- `1` if positive, `-1` if negative, and `0` if the number is a zero.
SELECT
  balance,
  SIGN(balance) AS sign
FROM account
-- | balance | sign |
-- | - | - |
-- | 102.21 | 1 |
-- | 0 | 0 |
-- | -122 | -1 |

-- ABS
-- Shows the absolute value of a signed number
SELECT ABS(balance) FROM account
```

## Temporal

In [ ]:
-- set time zone
SET time_zone = 'Europe/Zurich'
-- or
ALTER SESSION TIMEZONE = 'Europe/Zurich'

| Datatype | Description | Example |
| - | - | - |
| `TIMESTAMP` | Used by MySQL, PostgreSQL. Date format: `YYYY-MM-DD HH:MM:SS.MSS`. The TIMESTAMP data type has a range of '1970-01-01 00:00:01' UTC to '2038-01-09 03:14:07' UTC. It has varying properties, depending on the MySQL version and the SQL mode the server is running in. | A column that tracks when a user last modified a particular row in a table. |
| `DATETIME` | MySQL. Date format is like TIMESTAMP.  The DATETIME type is used when you need values that contain both date and time information. MySQL retrieves and displays DATETIME values in 'YYYY-MM-DD HH:MM:SS' format. The supported range is '1000-01-01 00:00:00' to '9999-12-31 23:59:59'. | |
| `DATE` | `YYYY-MM-DD` | Column to hold the expected future shipping date of a customer order. An employee's birth date. |
| `YEAR` | `YYYY` | |
| `TIME` | `HHH:MM:SS` | |

> Date is inserted as string in the format `YYYY-MM-DD`, e.g. `2020-03-23`. MySQL or other servers will automatically convert the string into a date, given that the format of the string matches that of the column in the temporal datatype

To automatically populate a table with the date that a record was inserted:
```sql
-- PostgreSQL, MySQL
CREATE TABLE table1 (
	name TEXT, 
--	date TIMESTAMP
	date TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
```

Different ways of writing time:
| Date | Example |
| - | - |
| `YYYY` | 2014 |
| `MM` | 01 to 12 |
| `DD` | 01 to 31 |
| `HH` | Hour: 00 to 23 |
| `HHH` | Hours (elapsed): -838 to 838 |
| `MI` | Minute: 00 to 59 |
| `SS` | Seconds: 00 to 59 |

**Time zones**

```sql
-- Give current UTC time
-- MySQL
SELECT utc_timestamp();

-- Check time zone settings - global time zone and session time zone
-- SYSTEM = server is using the time zone setting from the server on which the database resides
-- MySQL
SELECT @@global.time_zone, @@session.time_zone
-- Set time zone
SET time_zone = 'Europe/Zurich';
-- PostgreSQL
SHOW timezone;
SELECT current_setting('TIMEZONE');
```

**General**

```sql
-- Get a quarter number for a timestamp
SELECT quarter(datetime)

-- Update a date for a row
UPDATE rental
SET return_date = '2019-09-17' -- or '2019-09-17 15:30:00'
WHERE rental_id = 99999;

-- Gives YYYY-MM-DD HH:MM:SS.MSMS
SELECT NOW();
-- Get years of a person from his birthday
SELECT AGE(NOW(), date_of_birth);
-- Get current date; returns 2022-03-16
-- MySQL, CURDATE(); PostgreSQL, doesn't exist
SELECT CURDATE()
-- PostgreSQL, CURRENT_DATE;
-- MySQL, CURRENT_DATE or CURRENT_DATE()
select CURRENT_DATE() -- returns '2024-11-21'
-- Get current time; format `11:34:22`
SELECT CURTIME()
SELECT CURRENT_TIME()
-- Get current timestamp
SELECT CURRENT_TIMESTAMP
-- returns current date formatted as UNIX time
select UNIX_TIMESTAMP()
```

**DATE_SUB**

Subtract specified interval from the passed date argument.

```sql
-- Subtract 10 days from a date
SELECT DATE_SUB('2017-06-15', INTERVAL 10 DAY);
```

**FORMAT_DATE**

Format date in a specified format:
```sql
WITH temp0 AS (
  select '2025-01-01' rundate 
  union all select '2025-02-01' rundate
),

temp1 AS (
  select CAST(rundate AS DATE) rundate 
  from temp0
)

select
  rundate, 
  format_date('Q%Q', rundate) AS quarter, 
  format_date('%Y', rundate) AS year
FROM temp1
-- [{
--   "rundate": "2025-01-01",
--   "quarter": "Q1",
--   "year": "2025"
-- }, {
--   "rundate": "2025-02-01",
--   "quarter": "Q1",
--   "year": "2025"
-- }]
```



**Other**
```sql
-- Typecasting
-- YYYY-MM-DD
SELECT NOW()::DATE
-- HH:MM:SS.MSMS
SELECT NOW()::TIME
-- An example
(NOW()::DATE + INTERVAL '10 MONTHS')::DATE

-- STR_TO_DATE()
-- Change string format to specified datetime format
-- MySQL
-- Returns DATETIME, DATE, or TIME value depending on the contents of the format string
SELECT STR_TO_DATE('September 17, 2019', '%M %d, %Y'); -- generates DATE format: `YYYY-MM-DD`
```

**DAYNAME()**
- MySQL
- Extracts an entity from a date
- *Instead of this, it is better to use EXTRACT*
```sql
SELECT DAYNAME('2019-09-18') -- > Wednesday
```



```sql
-- Select a part of a date
-- year, month, day, hour, minute, second
SELECT date_part('year', (SELECT date_column_name))

-- Select year and month
SELECT TO_CHAR(order_date, 'YYYY-MM')




-- MySQL
-- if you have date containing minutes, hours, etc. apart from the date itself, you can filter only based on year,month,day like this:
WHERE return_date = date('2005-07-05') 
```

MySQL string formats to date type, e.g. `str_to_date('DEC-21-1980', '%b-%d-%Y')`:
| Formatter | Definition | Example |
| - | - | - |
| `%Y` | The four-digit year | |
| `%y` | The two-digit year | |
| `%M` | The full month name | (January..December) |
| `%m` | The numeric month | (0..12) |
| `%b` | The short month name | Jan, Feb, ... |
| `%W` | The full weekday name | (Sunday..Saturday) |
| `%w` | The numeric day of the week | (0=Sunday..6=Saturday) |
| `%a` | The short weekday name | Sun, Mon, ... |
| `%d` | The numeric day of the month | (01..31) |
| `%j` | The day of year | (001..366) |
| `%H` | The hour of the day, in 24-hour format | (00..23) |
| `%h` | The hour of the day, in 12-hour format | (01..12) |
| `%i` | The minutes within the hour | (00..59) |
| `%s` | The number of seconds | (00.59) |
| `%f` | The number of microseconds | (000000..999999) |
| `%p` | AM or PM | |

Examples:
```sql
-- return records where date equals to specified date
select * from personal_data where birthday = '1977-05-04'
select * from personal_data where birthday = '1977-05-04'::date;

-- Thus, we can order birthdays based only on month and date
-- For example, table like this:
--  id | name         |    date    |
-- ----+--------------+------------+
--  21 | Person 1     | 1971-11-21 |
--  23 | Person 2     | 1989-12-29 |

SELECT * 
FROM notable_dates 
ORDER BY 
  EXTRACT(MONTH FROM date) DESC, 
  EXTRACT(DAY FROM date) DESC;
```

Show total monthly payments for film rentals for 2005 for each quarter and month;
```sql
SELECT 
	QUARTER(payment_date) AS quarter,
	MONTHNAME(payment_date) AS month_nm,
	SUM(amount) AS monthly_sales,
	MAX(SUM(amount)) OVER () AS max_overall_sales,
	MAX(SUM(amount)) OVER (PARTITION BY QUARTER(payment_date)) AS max_qrtr_sales
FROM payment
WHERE YEAR(payment_date) = 2005
GROUP BY 
	quarter(payment_date),
	monthname(payment_date)
;

-- quarter|month_nm|monthly_sales|max_overall_sales|max_qrtr_sales|
-- -------+--------+-------------+-----------------+--------------+
--       2|May     |      4823.44|         28368.91|       9629.89|
--       2|June    |      9629.89|         28368.91|       9629.89|
--       3|July    |     28368.91|         28368.91|      28368.91|
--       3|August  |     24070.14|         28368.91|      28368.91|
```

## Type casting

You can cast datatypes in the ways below:
```sql
-- data types: date, numeric, int, float
-- does NOT work in BigQuery or MySQL,
-- but works in PostgreSQL
SELECT 
  whatever::date, 
  whatever2::numeric
-- or
round( SUM(rating::dec / position::dec)::dec / COUNT(rating)::dec, 2) AS quality

-- or
DATE('2025-01-05')

-- another way
SELECT 
  CAST(sss2.id AS STRING),
  CAST(age AS varchar),
  CAST('123' AS SIGNED INTEGER)
  CAST('2025-01-05' AS date)
```

Complex data types:
```sql
-- List - usually used within a WHERE _ IN <list> clause
('Value1', 'Value2', 'Value3')
```

## NULL

Different meanings / contexts of NULL:
- Not applicable: employee ID column for a transaction that took place at an ATM machine
- Value not yet known: federal ID is not known at the time a customer row is created
- Value undefined: when an account is created for a product that has not yet been added to the database

Rules:
- An expression can *be* null, but it can never *equal* null: 
  - ❌ `WHERE return_date = NULL` does not return any rows
  - ✅ `WHERE return_date IS NULL` or `IS NOT NULL`
- Two nulls are never equal to each other


## Array

BigQuery has two array indexing styles:
- `OFFSET(n)` - zero-based
- `ORDINAL(n)` - one-based.

Examples:

```sql
array[ORDINAL(2)] -- take the element at index 1
array[OFFSET(2)] -- take the element at index 2
```


## Array > column etc.

Convert sql array into rows of a column
```sql
SELECT *
FROM UNNEST(ARRAY[1, 2, 3,
                  4, 5, 6
]) AS id1
-- id1   |
-- ------+
--      1|
--      2|
--      3|
--      4|
--      5|
--      6|
```

Separate string into list items
```sql
-- Separate string into list items
SELECT string_to_array('1 2 3 4', ' ') -- gives you output of one cell like this: {1,2,3,4}
-- Separate and put as values of a column
SELECT unnest(string_to_array('1 2 3 4', ' '))
-- example usage: you have a string containing items you want to look for
select *
from employee
where emp_id in (
	select unnest(string_to_array('100 101 102', ' '))::numeric
)
-- make a database selection as a name
```


# Data type - specific functions

## Character

### LENGTH

> Works for MySQL and PostgreSQL
>
> Works only with string / character data types

For a specified column, returns the string length of each row

```sql
SELECT LENGTH(name)
FROM person;
-- returns:
-- length| -- int data type
-- ------+
--      9|
--      2|
--      5|
--      5|
```

In [ ]:
WITH temp1 AS (
	SELECT 'Manhattan' city
	UNION ALL
	SELECT 'Alaska' city
)
SELECT 
	city,
	LENGTH(city)
FROM temp1

-- city     |LENGTH(city)|
-- ---------+------------+
-- Manhattan|           9|
-- Alaska   |           6|

### SPLIT_PART

From a column with text values, where each cell has values with a delimiter, extract n-th value into a new column.

> Works in PostgreSQL

Simple example:
```sql
SELECT SPLIT_PART('a,b,c', ',', 2)
-- split_part|
-- ----------+
-- b         |
```

Another example, slightly more complex:
```sql
WITH temp1 AS (
	SELECT 'a1,b1,c1' AS str1
	UNION ALL
	SELECT 'a2,b2,c2' AS str1
)
SELECT 
	SPLIT_PART(str1, ',', 1) AS a,
	SPLIT_PART(str1, ',', 2) AS b,
	SPLIT_PART(str1, ',', 3) AS c
FROM temp1
-- a |b |c |
-- --+--+--+
-- a1|b1|c1|
-- a2|b2|c2|
```


### SUBSTRING

> In some dialects SUBSTR

Slice a string.

Basic syntax: `SUBSTRING(string, start_position, length)`


In [ ]:
-- example
WITH temp1 AS (
	SELECT 'Betty' name 
	UNION ALL 
	SELECT 'Anastasia' name
)
-- select substring from 1st to 3rd character
SELECT substring (name, 1, 3)
FROM temp1
-- substring|
-- ---------+
-- Bet      |
-- Ana      |


-- select substring from 6th to 14th character
SELECT SUBSTRING('this substring is great!', 6, 8+1)
-- substring


-- Using a negative start_position (in some dialects like MySQL): 
-- a negative start_position counts from the end of the string.
SELECT SUBSTRING('Hello, World!', -6); -- Result: 'World!'


### LEFT

Extract `n` characters from a string, starting from left:

```sql
SELECT LEFT('whatever', 3) AS ExtractString; -- output is `wha`

```

### character sets

**CHR**

> PostgreSQL

Based on ASCII code, return UTF-8 character. Check mapping here: https://www.languagetesting.com/windows-alt-codes

```sql
SELECT 
	CHR(97) || CHR(98) || CHR(99) || CHR(100), 
	CHR(126) || 'Buenos d' || CHR(237) || 'as, te gusta az' || CHR(250) || 'car?'

-- ?column?|?column?                      |
-- --------+------------------------------+
-- abcd    |~Buenos días, te gusta azúcar?|
```

**ASCII**

> Works for MySQL and PostgreSQL

Returns the ASCII code / number for a (UTF-8) character

```sql
SELECT ASCII('ñ') -- for instance, in character set UTF-8 it's a character 195


SELECT ASCII('í'), ASCII('ú')
-- ascii|ascii|
-- -----+-----+
--   237|  250|
```


### POSITION

In [ ]:
-- find the location of a substring within a string
SELECT POSITION('characters' IN ('hello characters'))

-- position|
-- --------+
--        7|


### QUOTE

> MySQL

Places quotes around the entire string and adds escapes to any single quotes / apostrophes within the string, normally to use this string in another program.

```sql
SELECT QUOTE(txt1)
FROM table1;
```



In [ ]:
SELECT QUOTE(person_id)
FROM person

-- output:
-- QUOTE(person_id)|
-- ---------------+
-- '58'           |
-- '92'           |
-- '182'          |
-- '118'          |


### TRIM

Removes spaces or specified characters from both ends of a string.

```sql
SELECT TRIM(name) FROM employees;
```

### UPPER

```sql
SELECT UPPER(name)
-- Capitalise the first letter only
SELECT CONCAT(
  UPPER(SUBSTRING(name,1,1)),
  LOWER(SUBSTRING(name, 2, LENGTH(name) - 1))
) AS name
```

### slicing

**Slicing a string**

> Index slicing a string
> Note that index begins with 1 here.

```sql
WITH temp1 AS (
  SELECT 'hello there' textvalue
)

SELECT SUBSTRING(textvalue, 5, 3)
FROM temp1
-- [{
--   "f0_": "o t"
-- }]
```


- MySQL, PostgreSQL
- For slicing
- Indexes in SQL start with 1

In [ ]:
-- Select substring from index 5 to index 10
SELECT SUBSTRING('yes hello world', 5, 5); -- returns 'hello'

-- 'hello' + '****' + 'world' -> 'hello****world'
SELECT CONCAT(
    SUBSTRING('yes hello world', 5, 5), 
    '****', 
    SUBSTRING('yes hello world', 11, 5) 
)


### REPLACE

Basic form:

`REPLACE(string_expression, string_pattern, string_replacement)`

Examples:
```sql
/* Simply substitute whitespaces with underscores
*/
WITH temp1 AS (
	SELECT 'John Wayne' AS full_name
	UNION ALL
	SELECT 'Mayor C Payne' AS full_name
)
SELECT REPLACE(full_name, ' ', '_')
FROM temp1
-- replace      |
-- -------------+
-- John_Wayne   |
-- Mayor_C_Payne|


/* In numbers, replace zeros with nothing
*/

WITH temp1 AS (
    SELECT 2010 AS salary
    UNION ALL
    SELECT 20 AS salary
    UNION ALL
    SELECT 215 AS salary
    UNION ALL
    SELECT 13 AS salary
)

SELECT 
	CAST(
		REPLACE (CAST(salary AS VARCHAR), '0', '')
	AS INT)
FROM temp1
-- replace|
-- -------+
--       21|
--       2|
--     215|
--      13|
```


### regular expressions

There are two ways of writing regular expressions in SQL:
- `LIKE`: simplified REGEXP; is not as powerful, but typically faster than regular expressions.
- `~` or `REGEXP`: True REGEXP

There are also some regex-like simple functions.

#### LEFT

First, very basic regex functions `LEFT`:

**LEFT**

> Works for PostgreSQL, MySQL

```sql
-- Match last names that begin with 'Q'
WHERE LEFT(last_name, 1) = 'Q' 
-- Match last names that begin with 'Qu'
WHERE LEFT(last_name, 2) = 'Qu'
```

#### LIKE




**LIKE**

> Works for MySQL, PostgreSQL

General form:
```sql
SELECT * 
FROM courses 
WHERE course LIKE '_lgorithms';
```

| Wildcard character | Meaning |
| --- | --- |
| `%` | any character, any number of times (including 0) |
| `_` | exactly 1 character |

These are used with the SQL keyword `LIKE`

```sql
-- Find any clients who are an LLC
client_name LIKE '%LLC';
-- Case insensitive
LOWER(client_name) LIKE 'david'
-- Find employees born in october
birth_date LIKE '____-10-%';

-- names starting with 'W'
LIKE 'W%'
-- the second letter is 'e'
LIKE '_e%'
-- values with a space in them
LIKE '% %'
-- Value ends with '.com'
LIKE '%.com';
-- last_name like MATTHEWS, WALTERS, WATTS
LIKE '_A_T%S'
-- negative LIKE
NOT LIKE '_lgorithms';
-- case-insensitive
ILIKE, NOT ILIKE
```

> `LIKE` Works for MySQL, PostgreSQL

The regex statement `LIKE` in the `SELECT` clause will return a boolean mask for whether a column matches that regexp.

```sql
SELECT 
  first_name,
  first_name LIKE 'A%' AS starts_with_a
  -- MySQL:      alternatively you can use: `first_name REGEXP '^A.*' AS starts_with_a`
  -- PostgreSQL: alternatively you can use: `first_name ~ '^A.*' AS starts_with_a`
FROM employee;
-- returns for MySQL
-- first_name|starts_with_a| -- bigint
-- ----------+-------------+
-- David     |0            |
-- Angela    |1            |
-- Kelly     |0            |
-- Stanley   |0            |
-- Andy      |1            |

-- returns for PostgreSQL
-- first_name|starts_with_a| -- bool
-- ----------+-------------+
-- David     |false        |
-- Angela    |true         |
-- Kelly     |false        |
-- Stanley   |false        |
-- Andy      |true         |
```


#### REGEXP

> MySQL: `REGEXP` 
> PostgreSQL: `~`
> BigQuery: `REGEXP_CONTAINS`

```sql
SELECT * FROM table1 WHERE name ~ '^Grandfather.+|.+parents.+'
-- Entries start with a vowel
WHERE CITY ~ '^[AEIOUaeiou].*';
WHERE CITY REGEXP '^[aeiou]';
-- Entries that do NOT start with a vowel
WHERE LOWER(CITY) NOT REGEXP '^[aeiou].+'

-- Ends with a vowel
WHERE CITY REGEXP '.+[aeiouAEIOU]$'

-- Does not start or end with a vowel
LOWER(CITY) NOT REGEXP '^[aeiou].+|.+[aeiou]$'
```

> Note that BigQuery works slightly different:

```sql
WITH temp1 AS (
  SELECT 'section: protein: 10%; fat: 5%; another section' descr1, 1 id 
  union all 
  select 'section: protein: 15%; fat: 7%; fibre: 10%' descr2, 2 id 
)
select *
from temp1
where
  REGEXP_CONTAINS(descr1, r'.+protein.+')

```


Note that regex can be also used in the `SELECT` clause:

```sql
WITH temp1 AS (
	SELECT 'Alyx' name UNION ALL 
	SELECT 'Brady' UNION ALL SELECT 'Carrie' UNION ALL SELECT 'Carry'
)

SELECT 
	name,
	name ~ 'y$'
FROM temp1

-- name  |?column?|
-- ------+--------+
-- Alyx  |false   |
-- Brady |true    |
-- Carrie|false   |
-- Carry |true    |
```

#### CONTAINS_SUBSTR

> Works only for BigQuery

```sql
SELECT *
FROM `database.countries`
WHERE CONTAINS_SUBSTR(country_name, 'rus')
-- will find rows which in column `country_name` have 'Cyprus', 'Belarus', 'Russia'
```

#### REGEXP_EXTRACT_ALL

> Works for BigQuery

In [ ]:
-- Example 1:

with temp1 AS (
  select 1 id, 'we are having a vegetarian snack that is also bio based as usual' search_string union all 
  select 2, 'suitable for vegetarians yes' union all 
  select 3, 'not suitable for anyone' union all 
  select 4, 'this product is bio-based yes' union all 
  select 5, 'the product is bio, yet based on artificial materials'
)

-- returns nested results if multiple matches
SELECT 
  id,
  REGEXP_EXTRACT_ALL(
    LOWER(search_string),
    'vegetarian|bio.based'
  ) AS found_matches
FROM temp1
-- [{
--   "id": "1",
--   "found_matches": ["vegetarian", "bio based"]
-- }, {
--   "id": "2",
--   "found_matches": ["vegetarian"]
-- }, {
--   "id": "3",
--   "found_matches": []
-- }, {
--   "id": "4",
--   "found_matches": ["bio-based"]
-- }, {
--   "id": "5",
--   "found_matches": []
-- }]

In [ ]:
-- Example 2:

with temp1 AS (
  select 1 id, 'we are having a vegetarian snack that is also bio based as usual' search_string union all 
  select 2, 'suitable for vegetarians yes' union all 
  select 3, 'not suitable for anyone' union all 
  select 4, 'this product is bio-based yes' union all 
  select 5, 'the product is bio, yet based on artificial materials'
),

keyword_match AS (
  SELECT 
    id,
    REGEXP_EXTRACT_ALL(
      LOWER(search_string),
      'vegetarian|bio.based'
    ) AS found_matches
  FROM temp1
)

SELECT DISTINCT
  found_matches_unnested AS keywords,
  id 
FROM keyword_match 
CROSS JOIN UNNEST(found_matches) AS found_matches_unnested
  
-- [{
--   "keywords": "vegetarian",
--   "id": "1"
-- }, {
--   "keywords": "bio based",
--   "id": "1"
-- }, {
--   "keywords": "vegetarian",
--   "id": "2"
-- }, {
--   "keywords": "bio-based",
--   "id": "4"
-- }]

In [ ]:
-- Example 3 - like example 2 but without cross join:

with temp1 AS (
  select 1 id, 'we are having a vegetarian snack that is also bio based as usual' search_string union all 
  select 2, 'suitable for vegetarians yes' union all 
  select 3, 'not suitable for anyone' union all 
  select 4, 'this product is bio-based yes' union all 
  select 5, 'the product is bio, yet based on artificial materials'
),

keyword_matches AS (
  SELECT 
    id,
    REGEXP_EXTRACT_ALL(
      LOWER(search_string),
      'vegetarian|bio.based'
    ) AS found_matches
  FROM temp1
)

SELECT
  id,
  found_matches_unnested
FROM 
  keyword_matches, 
  UNNEST(found_matches) AS found_matches_unnested

-- | id | found_matches_unnested |
-- | - | - |
-- | 1 | vegetarian |
-- | 1 | bio based |
-- | 2 | vegetarian |
-- | 4 | bio-based |


Note that this function only returns non-overlapping matches, for example, using this function to extract `ana` from `banana` returns only one substring, not two. Also, if one keyword is a subset of another, then one of them won't work unless you use `\\b`, but even then some edge cases won't work as expected - see the example below:

```sql
with temp1 AS (
  select 'this is a chemical free product' search_string
),
temp2 AS (
    SELECT
        search_string, 
        regexp_extract_all(
            LOWER(search_string),
            "\\bfree\\b|\\bchemical.free\\b"
            ) AS p_arr
    FROM temp1
)
select * 
from temp2

-- notice how it only matched "chemical free" and not just the keyword "free";
-- ideally, you would want both to be matched.

-- [{
--   "search_string": "this is a chemical free product",
--   "p_arr": ["chemical free"]
-- }]
```

### concatenation

There exist different way to concatenate in SQL:
- operator `||`
- function `CONCAT`:
  - Can handle any expression that returns a string;
  - In MySQL, will convert numbers, dates, etc. to string format


In [ ]:
SELECT 
	'one ' || 'sheep ' || 'slept.', 
	CONCAT('one ', 'sheep ', 'slept.')

-- ?column?        |concat          |
-- ----------------+----------------+
-- one sheep slept.|one sheep slept.|

In [ ]:
WITH temp1 AS (
	SELECT 1 id, 'John' name UNION ALL SELECT 2 id, 'Jane'
)
SELECT 
	temp1.name || temp1.id || ' : is a name.',
	CONCAT(temp1.name, temp1.id, ' : is a name')
FROM temp1

-- ?column?          |concat           |
-- ------------------+-----------------+
-- John1 : is a name.|John1 : is a name|
-- Jane2 : is a name.|Jane2 : is a name|

In [ ]:
-- Update a table's column by concatenating a string at the end
UPDATE table1
SET col1 = CONCAT(col1, ' additional string')

## Numeric

### ABS & SIGN

In [ ]:
WITH temp1 AS (
	SELECT 10 balance UNION ALL 
	SELECT 0 UNION ALL 
	SELECT -10
)

SELECT 
	balance,
	ABS(balance),
	SIGN(balance)
FROM temp1

-- balance|abs|sign|
-- -------+---+----+
--      10| 10| 1.0|
--       0|  0| 0.0|
--     -10| 10|-1.0|

### MOD

Calculates the remainder when one number is divided into another number.

In [ ]:
SELECT MOD(10, 4) -- -> 2
SELECT 10 % 4 -- -> 2

In [ ]:
-- Odd number
MOD(columnName, 2) <> 0
-- Even number
MOD(columnName, 2) = 0

### POW

In [ ]:
SELECT POW(2, 8) -- -> 2**8 = 256

### Rounding

In [ ]:
WITH temp1 AS (
	SELECT 1 number UNION ALL 
	SELECT 1.1 UNION ALL
	SELECT 1.11 UNION ALL 
	SELECT 1.111 UNION ALL 
	SELECT 1.9 UNION ALL 
	SELECT 1.99 UNION ALL 
	SELECT 1.999 UNION ALL 
	SELECT 9 UNION ALL 
	SELECT 11 UNION ALL 
	SELECT 17 UNION ALL
	SELECT 99 UNION ALL
	SELECT 109
)

SELECT 
	ROUND(number, -2) AS round_minus_2,
	ROUND(number, -1) AS round_minus_1,
	ROUND(number)     AS round_0,
	ROUND(number, 1)  AS round_1,
	FLOOR(number),
	CEIL(number),
	TRUNC(number, -2) AS trunc_minus_2,
	TRUNC(number, -1) AS trunc_minus_1,
	TRUNC(number)     AS trunc_0, -- is called TRUNCATE in some SQL systems 
	TRUNC(number, 1)  AS trunc_1
FROM temp1

-- round_minus_2|round_minus_1|round_0|round_1|floor|ceil|trunc_minus_2|trunc_minus_1|trunc_0|trunc_1|
-- -------------+-------------+-------+-------+-----+----+-------------+-------------+-------+-------+
--             0|            0|      1|    1.0|    1|   1|            0|            0|      1|    1.0|
--             0|            0|      1|    1.1|    1|   2|            0|            0|      1|    1.1|
--             0|            0|      1|    1.1|    1|   2|            0|            0|      1|    1.1|
--             0|            0|      1|    1.1|    1|   2|            0|            0|      1|    1.1|
--             0|            0|      2|    1.9|    1|   2|            0|            0|      1|    1.9|
--             0|            0|      2|    2.0|    1|   2|            0|            0|      1|    1.9|
--             0|            0|      2|    2.0|    1|   2|            0|            0|      1|    1.9|
--             0|           10|      9|    9.0|    9|   9|            0|            0|      9|    9.0|
--             0|           10|     11|   11.0|   11|  11|            0|           10|     11|   11.0|
--             0|           20|     17|   17.0|   17|  17|            0|           10|     17|   17.0|
--           100|          100|     99|   99.0|   99|  99|            0|           90|     99|   99.0|
--           100|          110|    109|  109.0|  109| 109|          100|          100|    109|  109.0|

## Temporal

In [ ]:
-- select current date and time at current location
select current_timestamp;

-- select current date and time at utc
SELECT current_timestamp AT time ZONE 'utc';

-- timezone               |
-- -----------------------+
-- 2026-03-26 21:03:05.458|

### Type casting

In [ ]:
-- CAST
-- is more strict
WITH temp1 AS (
	SELECT '2019-09-17 15:30:00' AS rundate
)

SELECT
	CAST(rundate AS timestamp),
	CAST(rundate AS date),
	CAST(rundate AS time)
FROM temp1

-- rundate                |rundate   |rundate |
-- -----------------------+----------+--------+
-- 2019-09-17 15:30:00.000|2019-09-17|15:30:00|

In [ ]:
-- TO_DATE (PostgreSQL)
-- str_to_date(MySQL)
SELECT to_date('September 17, 2019', 'Month DD, YYYY')

-- to_date   |
-- ----------+
-- 2019-09-17|

### INTERVAL

**INTERVAL**
- Is used to add time period to dates
- Used by PostgreSQL, MySQL
- Interval types (note: in PostgreSQL, can write as singular or plural - DAY or DAYS; in MySQL, only as singular - DAY): `second`, `minute`, `hour`, `day`, `month`, `year`, `minute_second`, `hour_second`, `year_month`


In [ ]:
-- Current date + 5 days

-- PostgreSQL 
SELECT CURRENT_DATE + INTERVAL '5 DAY'

-- MySQL
SELECT CURRENT_DATE() + INTERVAL 5 DAY;
SELECT DATE_ADD(CURRENT_DATE(), INTERVAL 5 DAY);


In [ ]:
-- Time a year ago: 
NOW() - INTERVAL '1 YEAR'; -- PostgreSQL

-- To current date add 1 year and 1 month
current_date + INTERVAL '1 year 1 month'

-- Select birthdays between 1977-05-04 and 30 days before that
SELECT * FROM personal_data 
WHERE birthday < '1977-05-04'::date 
AND birthday > '1977-05-04'::date - INTERVAL '30 DAYS';

In [ ]:
-- Add 3 hours, 27 minutes, and 11 seconds to the current timestamp

-- MySQL
SELECT DATE_ADD(CURRENT_TIMESTAMP(), INTERVAL '3:27:11' HOUR_SECOND)

-- PostgreSQL
SELECT CURRENT_TIMESTAMP + INTERVAL '3:27:11' HOUR_SECOND;

-- MySQL: add 9 years and 11 months to a birth_date
UPDATE employee
SET birth_date = DATE_ADD(birth_date, INTERVAL '9-11' YEAR_MONTH)
WHERE emp_id = 123;

### LAST_DAY

In [ ]:
-- MySQL, get last day of the month on the specified date
SELECT LAST_DAY('2019-09-17')

In [ ]:
-- PostgreSQL doesn't have an equivalent built-in function,
-- but you can use the following code
WITH temp1 AS (
	SELECT CAST('2019-09-17 15:30:00' AS timestamp) AS rundate
)

SELECT 
    (
        date_trunc('MONTH', rundate) 
        + INTERVAL '1 MONTH - 1 day'
    )::date AS end_of_month
FROM temp1

-- end_of_month|
-- ------------+
--   2019-09-30|

### DAYNAME

In [ ]:
-- MySQL, dayname - extract day from the date
SELECT dayname('2019-09-18')

In [ ]:
-- PostgreSQL
WITH temp1 AS (
	SELECT CAST('2019-09-17 15:30:00' AS timestamp) AS rundate
)

SELECT to_char(rundate, 'Day')
FROM temp1

-- to_char  |
-- ---------+
-- Tuesday  |

### EXTRACT


- Extracts fields: DAY, DOW, MONTH, YEAR, CENTURY
- EXTRACT can be used in SELECT and WHERE
- PostgreSQL, MySQL


In [ ]:
-- EXTRACT: Extracting fields: DAY, DOW, WEEK, MONTH, YEAR, CENTURY
-- EXTRACT can be used in SELECT and WHERE

-- MySQL
EXTRACT(YEAR FROM CURRENT_DATE())
-- MySQL
SELECT YEAR(NOW()), MONTH(NOW()), DAY(NOW()), HOUR(NOW()), MINUTE(NOW()), SECOND(NOW())
-- Select month of February
SELECT * FROM notable_dates WHERE EXTRACT (MONTH FROM date1) = 02
-- Compare two years
WHERE EXTRACT(YEAR FROM date1) < EXTRACT(YEAR FROM CURRENT_DATE())
WHERE EXTRACT(YEAR FROM e.birth_date) IN (1967, 1961)
-- MySQL
YEAR(date1) = 2004 -- or '2004'
QUARTER(date1)
MONTHNAME(date1)

In [ ]:
-- MySQL
WITH temp1 AS (
	SELECT CAST('2019-09-17 15:30:00' AS datetime) AS rundate
)

SELECT 
	EXTRACT(MONTH FROM rundate), -- 9
	MONTHNAME(rundate), -- September
    QUARTER(rundate), -- 3
    YEAR(rundate) -- 2019
FROM temp1

In [ ]:
-- PostgreSQL
WITH temp1 AS (
	SELECT CAST('2019-09-17 15:30:00' AS timestamp) AS rundate
)

SELECT 
	EXTRACT(YEAR FROM NOW()), -- 2026
	EXTRACT(DOW FROM rundate), -- 2
	EXTRACT(WEEK FROM rundate), -- 38
	TO_CHAR(rundate, 'FMDay'), -- Tuesday
	EXTRACT(YEAR FROM rundate), -- 2019
	EXTRACT(MONTH FROM rundate), -- 9
	TO_CHAR(rundate, 'Month'), -- September
	TO_CHAR(rundate, 'MONTH') -- SEPTEMBER
FROM temp1


### DATEDIFF

In [ ]:
-- DATEDIFF
-- MySQL
-- returns the number of full days between two dates
-- ignores the time of day in its arguments
SELECT DATEDIFF('2019-09-03', '2019-06-21') -- > 74
SELECT DATEDIFF('2019-09-03 23:59:59', '2019-06-21 00:00:01') -- > 74
SELECT DATEDIFF('2019-06-21', '2019-09-03') -- > -74
-- in PostgreSQL, you just subtract
SELECT '2025-09-20'::date - '2025-09-01'::date AS days_difference; -- -> 19 -- days
-- in BigQuery, slight variation
DATE_DIFF('2019-10-01', '2019-05-01', DAY)

# Aggregate functions

Aggregate statements / functions can be used in two ways:
- Implicit groups: there is no GROUP BY clause, so all rows are considered

    ```sql
    -- General form
    -- On their own
    SELECT 
    aggregate_function(column1) AS alias
    FROM table1


    SELECT
        SUM(amount)
    FROM table1;
    ```

- Explicit groups: used with GROUP BY clause. You specify over which group of rows the aggregated functions should be applied.

    ```sql
    -- With GROUP BY
    SELECT 
    column1, 
    aggregate_function(column2)
    FROM table1
    GROUP BY column1;


    SELECT
        person_id,
        SUM(amount)
    FROM table1
    GROUP BY person_id
    ```


> Note: aggregate functions such as AVG, MIN, and MAX cannot be used in a WHERE clause directly - they have to be wrapped in a subquery.


## MIN, MAX

In [ ]:
-- MIN, MAX
-- Print the max value of column2
SELECT MAX(column1)
-- Select the earliest date for each 'player_id' category
SELECT 
  player_id, 
  MIN(event_date) AS first_login
FROM Activity
GROUP BY player_id
-- Select the second highest salary in a table
SELECT MAX(salary) AS second_highest_salary
FROM employees
WHERE salary < (SELECT MAX(salary) FROM employees);

## AVG

In [ ]:
-- AVG
SELECT AVG(column1) 

## SUM

In [ ]:
-- SUM
-- Sum all values in a column
SELECT SUM(column1)
-- Find the total sales of each salesman
SELECT SUM(total_sales), emp_id FROM works_with GROUP BY emp_id;

## COUNT

- `COUNT(*)` counts all rows, including those with NULL values; 
- `COUNT(column1)` counts all rows, EXCEPT FOR NULL;
- Other functions such as SUM(column1), MAX(column1), AVG(column1) are also applied to all rows except for nulls;


In [ ]:
WITH temp1 AS (
  select 'John' name
  union all 
  select 'Jack' name
  union all
  select 'Jill' name
  union all 
  select NULL name
)

SELECT 
  -- Count the total number of rows
  COUNT(*) AS count_asterisk,
  -- Count non-null values in a column
  COUNT(name) AS count_name
FROM temp1

-- count_asterisk|count_name|
-- --------------+----------+
--              4|         3|

In [ ]:
-- COUNT
-- Count all female employees
SELECT COUNT(emp_id) FROM employee WHERE sex = 'F';

-- Count how many entries for each unique group in column 'sex' there are
SELECT COUNT(sex), sex FROM employee GROUP BY sex;

-- Count unique categories in a column
SELECT COUNT(DISTINCT sex) FROM employee;
SELECT COUNT(DISTINCT(sex)) FROM employee;

-- Equivalent of COUNTIF in excel
SUM(CASE WHEN state='approved' THEN 1 ELSE 0 END)

## STRING_AGG

> Note: string_agg = postgresql; group_concat = mysql;

```sql
-- STRING_AGG to concatenate strings
-- Below, the ORDER BY within the agg function is optional - it's just to sort the concatenated names lexicographically within each concatenation group
SELECT id, STRING_AGG(name, ', ' ORDER BY name) AS names
FROM some_table
GROUP BY id
```

> If you don't specify the separator, it will default to `,`: `STRING_AGG(column1)`

Concatenate all rows in column "countryname" into one cell, delimited by `, `
```sql
SELECT 
  STRING_AGG(countryname, ', ')
FROM table1
```

```sql
-- for each group in "supply_type", concatenate rows in the column "supplier_name" 
SELECT 
	supply_type,
	STRING_AGG(supplier_name, ' | ') AS suppliers_agg
FROM table1 
GROUP BY supply_type

-- supply_type     |suppliers_agg                            |
-- ----------------+-----------------------------------------+
-- Writing Utensils|Uni-ball | Uni-ball                      |
-- Paper           |Hammer Mill | Patriot Paper | Hammer Mill|
-- Custom Forms    |J.T. Forms & Labels | Stamford Lables    |

-- Same but with a non-textual column
STRING_AGG(CAST(supplier_id AS STRING), ' | ')
```

You can also concatenate rows from a non-textual column:
```sql
-- Method 1
-- works in PostgreSQL
WITH a1 AS (
	SELECT CAST(emp_id AS VARCHAR)
	FROM test.employee
)
SELECT STRING_AGG(emp_id, ' ') FROM a1
-- Method 2
SELECT STRING_AGG(CAST(emp_id AS VARCHAR), ' | ') -- VARCHAR for PostgreSQL and STRING for BigQuery
FROM test.employee
```

another example (you can comment out the `DISTINCT` statement below to see that you get repeats)
```sql
with temp1 AS (
  select 1 id, 'one' key, 'a' value
  union all 
  select 1 id, 'one' key, 'a' value
  union all 
  select 1 id, 'one' key, 'b' value
  union all 
  select 1 id, 'two' key, 'a' value
  union all
  select 2 id, 'one' key, 'a' value
  union all 
  select 2 id, 'two' key, 'b' value
  union all 
  select 2 id, 'two' key, 'b' value 
  union all 
  select 2 id, 'three' key, 'b' value
  union all 
  select 2 id, 'three' key, 'b' value
)
select 
  id,
  STRING_AGG(
    DISTINCT
      CONCAT(key, ' : ', value),
      ' || '
  ) AS descr_key_value
FROM temp1
group by id
-- [{
--   "id": "1",
--   "descr_key_value": "one : a || one : b || two : a"
-- }, {
--   "id": "2",
--   "descr_key_value": "one : a || two : b || three : b"
-- }]
```

You can also sort values within the `STRING_AGG` expression:

```sql
STRING_AGG(DISTINCT column1, ', ' ORDER BY column1)
```


Example of GROUP_CONCAT (MySQL):
```sql
-- concatenate all rows in the column "last_name" with separator ', '
group_concat(last_name ORDER BY first_name SEPARATOR ', ')
```


## Quantiles

`APPROX_QUANTILES`: 
- BigQuery = Aggregate functions

```sql
APPROX_QUANTILES([DISTINCT] expression, number [{IGNORE|RESPECT} NULLS])
```
- `expression`: the column or expression containing numeric data for which to calculate quantiles;
- `number`: an integer representing the number of quantiles to divide the data into, e.g. `4` for quartiles, `100` for percentiles;
- `IGNORE NULLS` or `RESPECT NULLS`: optional clauses to control null handling. By default, it's ignore nulls;

`PERCENTILE_DISC`: 
- Calculate percentiles, taking the value that exists in the dataset; so if there is an even number of data points, it takes the lower middle value
- BigQuery = window function 
- PostgreSQL = aggregate function

`PERCENTILE_CONT`:
- Find true quantiles: if there's an even number of data points, takes the average of the middle two values
- BigQuery = window function
- PostgreSQL = aggregate function


**BigQuery**

```sql
WITH table1 AS (
	SELECT 'Carpenter' AS profession, 70 AS age UNION ALL
	SELECT 'Carpenter', 50 UNION ALL
  SELECT 'Carpenter', 65 UNION ALL
  SELECT 'Carpenter', 45 UNION ALL
  SELECT 'Programmer', 30 UNION ALL
  SELECT 'Programmer', 35 UNION ALL
  SELECT 'Programmer', 20 UNION ALL
  SELECT 'Programmer', 25 
)
SELECT 
	profession,
	MIN(age) AS min_age,
	AVG(age) AS mean_age,
  -- BigQuery: calculate 10th and 90th percentile
	APPROX_QUANTILES(age, 100)[OFFSET(10)] AS percentile_10,
	APPROX_QUANTILES(age, 100)[OFFSET(90)] AS percentile_90,
FROM table1
GROUP BY 
	profession
```

In BigQuery, there is also a window function `PERCENTILE_CONT`:

```sql
WITH table1 AS (
	SELECT 'Carpenter' AS profession, 70 AS age UNION ALL
	SELECT 'Carpenter', 50 UNION ALL
  SELECT 'Carpenter', 65 UNION ALL
  SELECT 'Carpenter', 45 UNION ALL
  SELECT 'Programmer', 30 UNION ALL
  SELECT 'Programmer', 35 UNION ALL
  SELECT 'Programmer', 20 UNION ALL
  SELECT 'Programmer', 25 
)
SELECT
  profession, 
  age, 
  PERCENTILE_CONT(age, 0.95) OVER() AS p95_overall,
  PERCENTILE_CONT(age, 0.95) OVER(PARTITION BY profession) AS p95_per_profession
FROM table1
```


**PostgreSQL**

```sql
WITH table1 AS (
	SELECT 'Carpenter' AS profession, 70 AS age
	UNION ALL
	SELECT 'Carpenter' AS profession, 50 AS age
	UNION ALL
	SELECT 'Carpenter' AS profession, 65 AS age
	UNION ALL
	SELECT 'Carpenter' AS profession, 45 AS age
	UNION ALL
	SELECT 'Programmer' AS profession, 30 AS age
	UNION ALL
	SELECT 'Programmer' AS profession, 35 AS age
	UNION ALL
	SELECT 'Programmer' AS profession, 20 AS age
	UNION ALL
	SELECT 'Programmer' AS profession, 25 AS age
)
SELECT 
	profession,
	MIN(age) AS min_age,
	AVG(age) AS mean_age,
	percentile_disc(0.1) WITHIN GROUP(ORDER BY age) AS percentile_10_disc,
	percentile_disc(0.5) WITHIN GROUP(ORDER BY age) AS percentile_50_disc,
	percentile_disc(0.9) WITHIN GROUP(ORDER BY age) AS percentile_90_disc,
	percentile_cont(0.1) WITHIN GROUP(ORDER BY age) AS percentile_10_cont,
	percentile_cont(0.5) WITHIN GROUP(ORDER BY age) AS percentile_50_cont,
	percentile_cont(0.9) WITHIN GROUP(ORDER BY age) AS percentile_90_cont
FROM table1
GROUP BY 
	profession
```

# Window / analytic functions

> a.k.a. data windows, window functions, analytic function

In SQL, a window function allows advanced analytics by letting you calculate across rows while keeping all data intact; it is a function which uses values from one or multiple rows to return a value for each row. 
- ℹ️ This contrasts with an aggregate function, which returns a single value for multiple rows.

Window functions have an OVER clause; any function without an OVER clause is not a window function, but rather an aggregate or single-row (scalar) function.

Anatomy of a window function:
- Some aggregate function first;
- `OVER()` clause: defines the set of rows for the function. Every window function requires this;
- `PARTITION BY`: define data windows for analytic functions; split data into groups, applying the function within each group. Similar to GROUP BY but without collapsing rows;
- `ORDER BY`: orders rows within each partition, critical for functions like ROW_NUMBER() and RANK();

```sql
<function_name>() OVER (
  PARTITION BY <column> 
  ORDER BY <column>
)
```

In [ ]:
-- example 0
WITH temp1 AS (
  SELECT 100 price, 1 id 
  union all 
  select 150 price, 2 id 
  union all
  select 200 price, 2 id 
)
select
  id, 
  price, 
  SUM(price) OVER (PARTITION BY id) AS sum_price
FROM temp1
-- Fila	id	price   sum_price
-- 1	1	100     100
-- 2	2	150     350
-- 3	2	200     350



-- example 1
-- Show total monthly payments for film rentals for 2005 for each quarter and month;
SELECT 
	QUARTER(payment_date) AS quarter,
	MONTHNAME(payment_date) AS month_nm,
	SUM(amount) AS monthly_sales,
	MAX(SUM(amount)) OVER () AS max_overall_sales,
	MAX(SUM(amount)) OVER (PARTITION BY QUARTER(payment_date)) AS max_qrtr_sales
FROM payment
WHERE YEAR(payment_date) = 2005
GROUP BY 
	quarter(payment_date),
	monthname(payment_date)
;
-- quarter|month_nm|monthly_sales|max_overall_sales|max_qrtr_sales|
-- -------+--------+-------------+-----------------+--------------+
--       2|May     |      4823.44|         28368.91|       9629.89|
--       2|June    |      9629.89|         28368.91|       9629.89|
--       3|July    |     28368.91|         28368.91|      28368.91|
--       3|August  |     24070.14|         28368.91|      28368.91|



-- example 2
SELECT 
	year,
	month, 
	MAX(MAX(sale)) OVER () AS max_overall_sale,
	MAX(SUM(sale)) OVER (PARTITION BY month) AS total_monthly_sales,
	MAX(MAX(sale)) OVER (PARTITION BY year, month) AS max_monthly_sales
FROM (
	SELECT '2014' AS year, 'June' AS month, 100 AS sale
	UNION ALL
	SELECT '2014' AS year, 'June' AS month, 150 AS sale
	UNION ALL
	SELECT '2014' AS year, 'July' AS month, 170 AS sale
	UNION ALL
	SELECT '2015' AS year, 'June' AS month, 300 AS sale
	UNION ALL
	SELECT '2015' AS year, 'June' AS month, 50 AS sale
) a1
GROUP BY 
	year,
	month

-- Result: 
-- year|month|max_overall_sale|total_monthly_sales|max_monthly_sales|
-- ----+-----+----------------+-------------------+-----------------+
-- 2014|July |             300|                170|              170|
-- 2014|June |             300|                350|              150|
-- 2015|June |             300|                350|              300|



-- example
-- using PARTITION BY and ORDER BY in the window function
-- get a different set of rankings per quarter
SELECT
	QUARTER(payment_date) quarter,
	MONTH(payment_date) month_nm,
	SUM(amount) monthly_sales,
	RANK() OVER (
		PARTITION BY QUARTER(payment_date) 
		ORDER BY SUM(amount) DESC
		) qtr_sales_rank
FROM payment
GROUP BY 1, 2
ORDER BY 1, 2


#### RANK

Ranking functions:
- `ROW_NUMBER`: returns a unique number for each row, with rankings arbitrarily assigned in case of a tie;
- `RANK`: returns the same ranking in case of a tie, with gaps in the ranking;
- `DENSE_RANK`: returns the same ranking in case of a tie, with NO gaps in the ranking

> For many situations, the `RANK` function might be the best option


In [ ]:
-- A clear comparative example
SELECT 
	customer_id,
	num_rentals,
	ROW_NUMBER() OVER (ORDER BY num_rentals DESC) AS row_number_rnk,
	RANK() OVER (ORDER BY num_rentals DESC) AS rank_rnk,
	DENSE_RANK() OVER (ORDER BY num_rentals DESC) AS dense_rank_rnk
FROM (	
	SELECT 1 customer_id, 46 num_rentals
	UNION ALL
	SELECT 2 customer_id, 45 num_rentals
	UNION ALL
	SELECT 3 customer_id, 45 num_rentals
	UNION ALL
	SELECT 4 customer_id, 44 num_rentals
	UNION ALL
	SELECT 5 customer_id, 44 num_rentals
	UNION ALL
	SELECT 6 customer_id, 44 num_rentals
	UNION ALL
	SELECT 7 customer_id, 43 num_rentals
) a1;
-- customer_id|num_rentals|row_number_rnk|rank_rnk|dense_rank_rnk|
-- -----------+-----------+--------------+--------+--------------+
--           1|         46|             1|       1|             1|
--           2|         45|             2|       2|             2|
--           3|         45|             3|       2|             2|
--           4|         44|             4|       4|             3|
--           5|         44|             5|       4|             3|
--           6|         44|             6|       4|             3|
--           7|         43|             7|       7|             4|


In [ ]:
-- Set rankings for each month
SELECT
    customer_id,
    rental_month,
    COUNT(*) num_rentals
    RANK() OVER (
        PARTITION BY rental_month
        ORDER BY COUNT(*) DESC
        ) rank_rnk
FROM rental
GROUP BY customer_id, rental_month
ORDER BY 2, 3 DESC;

In [ ]:
-- For each group, save only the longest string
WITH a1 AS (
	SELECT 'Text 1' AS texts, 1 AS groups
	UNION ALL
	SELECT 'Longer texts' AS texts, 1 AS groups
	UNION ALL
	SELECT 'fasdd asdfasd' AS texts, 2 AS groups
	UNION ALL
	SELECT 'aas algitorkdm' AS texts, 2 AS groups
	UNION ALL 
	SELECT NULL AS texts, 3 AS groups
	UNION ALL
	SELECT 'aaa' AS texts, 3 AS groups
),
a2 AS (
	SELECT 
		groups,
		CASE WHEN texts IS NULL THEN '-' ELSE texts END AS texts
	FROM a1
),
a3 AS (
	SELECT
		texts, 
		groups,
		ROW_NUMBER() OVER (PARTITION BY groups ORDER BY LENGTH(texts) DESC)
	FROM a2
)
SELECT *
FROM a3
WHERE row_number = 1



-- Get top 3 for ROW_NUMBER() function:
SELECT *
FROM (
	SELECT 
		customer_id,
		num_rentals,
		ROW_NUMBER() OVER (ORDER BY num_rentals DESC) AS row_number_rnk
	FROM (	
		SELECT 1 customer_id, 46 num_rentals
		UNION ALL
		SELECT 2 customer_id, 45 num_rentals
		UNION ALL
		SELECT 3 customer_id, 45 num_rentals
		UNION ALL
		SELECT 4 customer_id, 44 num_rentals
		UNION ALL
		SELECT 5 customer_id, 44 num_rentals
		UNION ALL
		SELECT 6 customer_id, 44 num_rentals
		UNION ALL
		SELECT 7 customer_id, 43 num_rentals
	) a1
) a2
WHERE 
	row_number_rnk <= 3
;
-- customer_id|num_rentals|row_number_rnk|
-- -----------+-----------+--------------+
--           1|         46|             1|
--           2|         45|             2|
--           3|         45|             3|



-- Multiple ranking - get ranking for each year
WITH temp1 AS (
	SELECT 1 customer_id, 2014 yr, 46 num_rentals
	UNION ALL
	SELECT 2 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 3 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 4 customer_id, 2014 yr, 44 num_rentals
	UNION ALL
	SELECT 5 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 6 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 7 customer_id, 2015 yr, 43 num_rentals
)
SELECT 
	customer_id,
	yr,
	num_rentals,
	ROW_NUMBER() OVER (PARTITION BY yr ORDER BY num_rentals DESC) AS num_rentals_rank
FROM temp1

-- customer_id|yr  |num_rentals|dense_rank_rnk|
-- -----------+----+-----------+--------------+
--           1|2014|         46|             1|
--           2|2014|         45|             2|
--           3|2014|         45|             2|
--           4|2014|         44|             3|
--           5|2015|         44|             1|
--           6|2015|         44|             1|
--           7|2015|         43|             2|

#### QUALIFY

QUALIFY is a clause used to filter the results of a window function. 

> You need a QUALIFY statement because WHERE, GROUP BY, and HAVING filtering statements are all evaluated before the window functions. This can be overcome either by QUALIFY within the same query or writing filters on a window function-containing query contained within a CTE.

Examples:
> QUALIFY is available in Snowflake, Databricks, ClickHouse, BigQuery, Redshift
> QUALIFY doesn't work in PostgreSQL or MySQL

**Basic usage example**:

```sql
/* This is the way you filter the lowest num_rentals
per year, using CTEs
*/
WITH temp1 AS (
	SELECT 1 customer_id, 2014 yr, 46 num_rentals
	UNION ALL
	SELECT 2 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 3 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 4 customer_id, 2014 yr, 44 num_rentals
	UNION ALL
	SELECT 5 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 6 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 7 customer_id, 2015 yr, 43 num_rentals
),

ranked_orders AS (
	SELECT
		*,
		ROW_NUMBER() OVER (
			PARTITION BY yr
			ORDER BY num_rentals ASC 
		) AS rn 
	FROM temp1
)

SELECT *
FROM ranked_orders
WHERE rn = 1;

-- customer_id|yr  |num_rentals|rn|
-- -----------+----+-----------+--+
--           4|2014|         44| 1|
--           7|2015|         43| 1|

/* However, you can do the same as above 
in one step (instead of two) 
using QUALIFY statement
*/
WITH temp1 AS (
	SELECT 1 customer_id, 2014 yr, 46 num_rentals
	UNION ALL
	SELECT 2 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 3 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 4 customer_id, 2014 yr, 44 num_rentals
	UNION ALL
	SELECT 5 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 6 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 7 customer_id, 2015 yr, 43 num_rentals
)
SELECT *
FROM temp1
QUALIFY ROW_NUMBER() OVER(
	PARTITION BY yr 
	ORDER BY num_rentals ASC
) = 1
```


```sql
-- Notice in this example that QUALIFY is applied AFTER 
-- apllying the WHERE clause
WITH temp1 AS (
	SELECT 1 customer_id, 2014 yr, 46 num_rentals
	UNION ALL
	SELECT 2 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 3 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 4 customer_id, 2014 yr, 44 num_rentals
	UNION ALL
	SELECT 5 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 6 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 7 customer_id, 2015 yr, 43 num_rentals
)
SELECT *
FROM temp1
WHERE customer_id IN (1, 2, 5, 6)
QUALIFY ROW_NUMBER() OVER(
	PARTITION BY yr 
	ORDER BY num_rentals ASC
) = 1
-- | customer_id | yr   | num_rentals |
-- | ----------- | ---- | ----------- |
-- | 2           | 2014 | 45          |
-- | 5           | 2015 | 44          |

-- if you comment out the WHERE clause, then apply QUALIFY in a separate CTE, and then apply WHERE, 
-- the results will be different - will return empty table
```

#### Reporting functions (MIN, MAX, SUM, etc.)

In [ ]:
-- generate monthly and grand totals for all payments
SELECT
    payment_month,
    amount,
    SUM(amount) OVER (
        PARTITION BY payment_month
        ) monthly_total,
    SUM(amount) OVER () grand_total
FROM payment
ORDER BY 1;
| payment_month | amount | monthly_total | grand_total |
| ------------- | ------ | ------------- | ----------- |
| August        | 10     | 30            | 60          |
| August        | 10     | 30            | 60          |
| August        | 10     | 30            | 60          |
| July          | 10     | 20            | 60          |
| July          | 10     | 20            | 60          |
| June          | 10     | 10            | 60          |

#### LAG, LEAD

- `LAG`: while processing current row's value, get previous row's value
- `LEAD`: same but for next row

In [ ]:
WITH temp1 AS (
	SELECT 'Lisa' name, '2021-01-01' date, 5500 salary
	UNION ALL
	SELECT 'Lisa' name, '2022-01-01' date, 7000 salary
	UNION ALL
	SELECT 'Lisa' name, '2023-01-01' date, 7500 salary
	UNION ALL
	SELECT 'Lisa' name, '2024-01-01' date, 8000 salary
)
SELECT
	name, 
	date,
	salary,
	LAG(salary) OVER (ORDER BY date) AS prev_salary,
	LEAD(salary) OVER (ORDER BY date) AS next_salary,
	100 * (salary - LAG(salary) OVER (ORDER BY date) ) / LAG(salary) OVER (ORDER BY date) AS percent_salary_increase
FROM temp1;

-- name|date      |salary|prev_salary|next_salary|percent_salary_increase|
-- ----+----------+------+-----------+-----------+-----------------------+
-- Lisa|2021-01-01|  5500|           |       7000|                       |
-- Lisa|2022-01-01|  7000|       5500|       7500|                     27|
-- Lisa|2023-01-01|  7500|       7000|       8000|                      7|
-- Lisa|2024-01-01|  8000|       7500|           |                      6|


#### Cumulative SUM

In [ ]:
-- method 1
WITH daily_sales AS (
	SELECT '2023-05-01'::date date, 100 sales
	UNION ALL
	SELECT '2023-05-02'::date date, 200 sales
	UNION ALL
	SELECT '2023-05-03'::date date, 150 sales
)
SELECT 
	date, 
	sales,
	SUM(sales) OVER (ORDER BY Date) as cumulative_sum
FROM daily_sales;
-- date      |sales|cumulative_sum |
-- ----------+-----+---------------+
-- 2023-05-01|  100|            100|
-- 2023-05-02|  200|            300|
-- 2023-05-03|  150|            450|


-- method 2 - same result
WITH daily_sales AS (
	SELECT '2023-05-01'::date date, 100 sales
	UNION ALL
	SELECT '2023-05-02'::date date, 200 sales
	UNION ALL
	SELECT '2023-05-03'::date date, 150 sales
)

SELECT
	date,
	sales,
	SUM(sales) OVER (ORDER BY date ROWS UNBOUNDED PRECEDING) AS cumulative_sum
FROM daily_sales
ORDER BY 1
-- date      |sales|cumulative_sum |
-- ----------+-----+---------------+
-- 2023-05-01|  100|            100|
-- 2023-05-02|  200|            300|
-- 2023-05-03|  150|            450|

In [ ]:
-- cumulative sum but with GROUP BY
-- works with MySQL, PostgreSQL
WITH yearly_sales AS (
	SELECT 2020 date_year, 100 sales UNION ALL 
	SELECT 2020 date_year, 100 sales UNION ALL 
	SELECT 2021 date_year, 150 sales UNION ALL
	SELECT 2022 date_year, 200 sales UNION ALL 
	SELECT 2022 date_year, 250 sales
)

SELECT
	date_year,
	SUM(sales) year_total,
	SUM(SUM(sales)) OVER (ORDER BY date_year ROWS UNBOUNDED PRECEDING) AS cumulative_sum
FROM yearly_sales
GROUP BY date_year
ORDER BY 1


#### Rolling average

> a.k.a.: rolling average, sliding window

In [ ]:
-- calculate a three-week rolling average of sales
WITH yearly_sales AS (
	SELECT 2020 date_year, 200 sales UNION ALL 
	SELECT 2021 date_year, 150 sales UNION ALL
	SELECT 2022 date_year, 450 sales
)

SELECT
	date_year,
	sales,
	AVG(sales) OVER (ORDER BY date_year ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING) AS rollign_3yr_avg
FROM yearly_sales
ORDER BY 1

-- date_year|sales|rollign_3yr_avg     |
-- ---------+-----+--------------------+
--      2020|  200|175.0000000000000000|
--      2021|  150|266.6666666666666667|
--      2022|  450|300.0000000000000000|

In [ ]:
-- calculate a three-week rolling average of sales
-- but combine GROUP BY with a window function
WITH yearly_sales AS (
	SELECT 2020 date_year, 100 sales UNION ALL 
	SELECT 2020 date_year, 100 sales UNION ALL 
	SELECT 2021 date_year, 150 sales UNION ALL
	SELECT 2022 date_year, 200 sales UNION ALL 
	SELECT 2022 date_year, 250 sales
)

SELECT
	date_year,
	SUM(sales) year_total,
	AVG(SUM(sales)) OVER (ORDER BY date_year ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING) AS rollign_3yr_avg
FROM yearly_sales
GROUP BY date_year
ORDER BY 1



In [ ]:
-- rolling 3-day average, but strictly 3 consecutive days average (takes gaps between days into account)
WITH daily_sales AS (
	SELECT '2020-01-01'::date date1, 100 sales
	UNION ALL
	SELECT '2020-01-02'::date date1, 200 sales
	UNION ALL
	SELECT '2020-01-03'::date date1, 150 sales
	UNION ALL 
	SELECT '2020-01-05'::date, 1000 UNION ALL 
	SELECT '2020-01-06'::date, 1500 UNION ALL
	SELECT '2020-01-07'::date, 1700
)

SELECT 
	date1,
	SUM(sales),
	AVG(SUM(sales)) OVER (ORDER BY date1 RANGE BETWEEN INTERVAL '1 DAY' PRECEDING AND INTERVAL '1 DAY' FOLLOWING) rolling_3_day_avg
FROM daily_sales
GROUP BY date1

In [ ]:
-- E.g. this leet code question: https://leetcode.com/problems/restaurant-growth/submissions/1649171860/?envType=study-plan-v2&envId=top-sql-50

-- Query below answer the question: 
-- - What is the cumulative sum of current day with the previous day for purchases? 
-- - What is the average purchase between each day and it's previous day? 

WITH temp1 AS (
  SELECT '2024-01-01' date, 10 purchase
  UNION ALL
  SELECT '2024-01-02' date, 50 purchase
  UNION ALL
  SELECT '2024-01-02' date, 50 purchase
  UNION ALL
  SELECT '2024-01-03' date, 60 purchase
  UNION ALL
  SELECT '2024-01-04' date, 70 purchase
  UNION ALL
  SELECT '2024-01-04' date, 10 purchase
), 
temp2 AS (
  SELECT 
    CAST(date AS date) AS date,
    purchase
  FROM temp1
)

SELECT 
  date,
  purchase,
  SUM(purchase) OVER (
    ORDER BY date 
    RANGE BETWEEN INTERVAL 1 DAY PRECEDING 
    AND CURRENT ROW 
  ) AS cumsum_prev_day,
  ROUND(
    AVG(purchase) OVER (
      ORDER BY date 
      RANGE BETWEEN INTERVAL 1 DAY PRECEDING 
      AND CURRENT ROW)
  , 2) AS cumavg_prev_day
FROM temp2


#### FIRST_VALUE

FIRST_VALUE() returns the first value in an ordered partition, while LAST_VALUE() returns the last value. 

𝗙𝗼𝗿 𝗲𝘅𝗮𝗺𝗽𝗹𝗲: In analyzing stock prices, FIRST_VALUE() can be used to compare daily stock prices to the price at month's start, so we can measure price changes relative to the month's opening price.

Example:

**FIRST_VALUE() behaves exactly as expected:**
```sql
WITH temp1 AS (
	SELECT 1 customer_id, 2014 yr, 46 num_rentals
	UNION ALL
	SELECT 2 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 3 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 4 customer_id, 2014 yr, 44 num_rentals
	UNION ALL
	SELECT 5 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 6 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 7 customer_id, 2015 yr, 43 num_rentals
)
SELECT 
	customer_id,
	yr,
	num_rentals,
	FIRST_VALUE(num_rentals) OVER (PARTITION BY yr ORDER BY num_rentals)
FROM temp1;
-- customer_id|yr  |num_rentals|first_value|
-- -----------+----+-----------+-----------+
--           4|2014|         44|         44|
--           2|2014|         45|         44|
--           3|2014|         45|         44|
--           1|2014|         46|         44|
--           7|2015|         43|         43|
--           5|2015|         44|         43|
--           6|2015|         44|         43|

```

**LAST_VALUE(), however, has an interesting case:**

```sql
/*
If you run it like below, it will find the last row between the start and the current row for each partition
*/
WITH temp1 AS (
	SELECT 1 customer_id, 2014 yr, 46 num_rentals
	UNION ALL
	SELECT 2 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 3 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 4 customer_id, 2014 yr, 44 num_rentals
	UNION ALL
	SELECT 5 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 6 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 7 customer_id, 2015 yr, 43 num_rentals
)
SELECT 
	customer_id,
	yr,
	num_rentals,
	LAST_VALUE(num_rentals) OVER (PARTITION BY yr ORDER BY num_rentals)
FROM temp1
-- customer_id|yr  |num_rentals|last_value|
-- -----------+----+-----------+----------+
--           4|2014|         44|        44|
--           2|2014|         45|        45|
--           3|2014|         45|        45|
--           1|2014|         46|        46|
--           7|2015|         43|        43|
--           5|2015|         44|        44|
--           6|2015|         44|        44|

/*
This is because by default, if you don't specify the range for the LAST_VALUE function, it runs the following: 
LAST_VALUE(num_rentals) OVER (PARTITION BY yr ORDER BY num_rentals RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
which exactly finds the last value in the range between the starting value and the current row's value.

Nevertheless, if you (as expected) want to find the last row in the range of starting value to the end value of a partition, 
you need to specify it in the range:
*/
WITH temp1 AS (
	SELECT 1 customer_id, 2014 yr, 46 num_rentals
	UNION ALL
	SELECT 2 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 3 customer_id, 2014 yr, 45 num_rentals
	UNION ALL
	SELECT 4 customer_id, 2014 yr, 44 num_rentals
	UNION ALL
	SELECT 5 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 6 customer_id, 2015 yr, 44 num_rentals
	UNION ALL
	SELECT 7 customer_id, 2015 yr, 43 num_rentals
)
SELECT 
	customer_id,
	yr,
	num_rentals,
	LAST_VALUE(num_rentals) OVER (PARTITION BY yr ORDER BY num_rentals RANGE BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)
FROM temp1
-- customer_id|yr  |num_rentals|last_value|
-- -----------+----+-----------+----------+
--           4|2014|         44|        46|
--           2|2014|         45|        46|
--           3|2014|         45|        46|
--           1|2014|         46|        46|
--           7|2015|         43|        44|
--           5|2015|         44|        44|
--           6|2015|         44|        44|
```

# Outliers handling

##### Quantiles

**BigQuery**

```sql
-- Outlier handling based on 5th and 95th quantile
WITH t_temp AS (
	SELECT 'Carpenter' AS profession, 70 AS age UNION ALL
	SELECT 'Carpenter', 50 UNION ALL
  SELECT 'Carpenter', 65 UNION ALL
  SELECT 'Carpenter', 45 UNION ALL
  SELECT 'Programmer', 30 UNION ALL
  SELECT 'Programmer', 35 UNION ALL
  SELECT 'Programmer', 20 UNION ALL
  SELECT 'Programmer', 25 
),

t_percentile AS (
  SELECT 
    *,
    PERCENTILE_CONT(age, 0.95) OVER(PARTITION BY profession) AS percentile_95,
    PERCENTILE_CONT(age, 0.05) OVER(PARTITION BY profession) AS percentile_5
  FROM t_temp
)

SELECT
  * EXCEPT (percentile_95, percentile_5)
FROM t_percentile
WHERE
  age > percentile_5
  AND age < percentile_95
/* It just removes ages 70 and 45 from `Carpenter`
and 20 and 35 from `Programmer`
*/
```

However, the problem with using quantiles for removing outliers is that it will remove any values from top and bottom, even if they are actually very close: 
```sql
WITH t_temp AS (
	SELECT 'Carpenter' AS profession, 1 AS age UNION ALL
	SELECT 'Carpenter', 2 UNION ALL
  SELECT 'Carpenter', 3 UNION ALL
  SELECT 'Carpenter', 4 UNION ALL
  SELECT 'Programmer', 10 UNION ALL
  SELECT 'Programmer', 11 UNION ALL
  SELECT 'Programmer', 12 UNION ALL
  SELECT 'Programmer', 13 
),

t_percentile AS (
  SELECT 
    *,
    PERCENTILE_CONT(age, 0.95) OVER(PARTITION BY profession) AS percentile_95,
    PERCENTILE_CONT(age, 0.05) OVER(PARTITION BY profession) AS percentile_5
  FROM t_temp
)

SELECT
  * EXCEPT (percentile_95, percentile_5)
FROM t_percentile
WHERE
  age > percentile_5
  AND age < percentile_95
-- this will remove rows with ages 1 and 4 from `Carpenter`
-- and rows with ages 10 and 13 from `Programmer`
```

##### Z-score

```sql
WITH data1 AS (
	SELECT 1 category, 'protein' nutrient, 3 value
	UNION ALL 
	SELECT 1, 'protein', 5
	UNION ALL 
	SELECT 1, 'protein', 19
	UNION ALL 
	SELECT 1, 'protein', 9
	UNION ALL 
	SELECT 1, 'protein', 100
	UNION ALL 
	SELECT 1, 'fat', 7
	UNION ALL
	SELECT 1, 'fat', 8
	UNION ALL
	SELECT 1, 'fat', 9
	UNION ALL
	SELECT 1, 'fat', 39
	UNION ALL
	SELECT 1, 'fat', 2
	UNION ALL
	SELECT 2, 'protein', 15
	UNION ALL
	SELECT 2, 'protein', 10
	UNION ALL
	SELECT 2, 'protein', 1000
	UNION ALL
	SELECT 2, 'fat', 14
	UNION ALL
	SELECT 2, 'fat', 19
), 

intermediate_calc AS (
	SELECT
		category,
		nutrient,
		value,
		AVG(value) OVER (PARTITION BY category, nutrient) AS mu,
		COUNT(value) OVER (PARTITION BY category, nutrient) AS n,
		(value - AVG(value) OVER (PARTITION BY category, nutrient))
			* (value - AVG(value) OVER (PARTITION BY category, nutrient))
			AS x_minus_mu_squared
	FROM data1
),

calculate_sigma AS (
	SELECT
		category,
		nutrient,
		value,
		mu,
		n, 
		SQRT(SUM(x_minus_mu_squared) OVER (PARTITION BY category, nutrient) / n) AS sigma
	FROM intermediate_calc
)

SELECT
	category,
	nutrient,
	(value - mu) / sigma AS zscore
FROM calculate_sigma
ORDER BY category ASC, nutrient ASC
```

# Sampling

## Random

https://render.com/blog/postgresql-random-samples-big-tables

**random()**


First type of sort is this. 

Intuitive but very inefficient.

```sql
SELECT * FROM my_events -- first, examines every row in the table
ORDER BY random() -- performs a lot of comparisons to sort
LIMIT 10000;
```

Next type - Bernoulli sampling. Much faster (as you just go through the data once) but the output is non-deterministic in the count of rows you get.

```sql
-- Random returns a value in the range [0, 1)
-- Therefore we compare against (0.001% / 100) to get ~10k rows
SELECT * FROM sample_values WHERE random() < 0.00001;
```


**TABLESAMPLE SYSTEM**

Sample N % of all data points. 

> More info: https://cloud.google.com/bigquery/docs/table-sampling

```sql
-- IMPORTANT! sampling is always done before the filtering. Here, you will first sample the table and then to that sample apply the WHERE clause, so the actual amount of sampled data will vary 
SELECT *
FROM table1 TABLESAMPLE SYSTEM (1)
WHERE last_name = 'Wayne'

-- or
SELECT *
FROM table1 AS t1
TABLESAMPLE SYSTEM (0.1) -- sample 0.1% of rows
WHERE last_name = 'Wayne'

-- To first filter and then do sampling you can do this, you can create a temporary table but I don't know how to do it: https://dba.stackexchange.com/questions/258271/perform-tablesample-with-where-clause-in-postgresql#:~:text=However%20you%20can%20work%20around%20this%20if%20you%20really%20want%20to%20use%20the%20tablesample%20attribute%20by%20creating%20a%20temporary%20table%20(or%20similar)%20based%20on%20your%20conditional%20query.

```

## FARM_FINGERPRINT

> Only works in BigQuery

Computer the fingerprint of a STRING or BYTES value, using the FarmHash Fingerprint64 algorithm. The output of this function for a particular input will never change. **It is a fingerprint function, whose requirement it is to simply produce a deterministic unique hash value for every unique input (avoid collisions)**. 

Example:
```sql
SELECT 
  id,
  FARM_FINGERPRINT(name)
FROM `dataset.table1`
```

This is useful for making reproducible samples. For example, in the query below, since each unique value processed with farm_fingerprint maps to a unique fingerprint (reproducibly), then this can serve as a proxy for a random, but reproducible sample. 
```sql
WITH temp1 AS (
	SELECT 1 id, 103 value
	UNION ALL
	SELECT 2 id, 102 value
	UNION ALL
	SELECT 3 id, 1339 value
	UNION ALL
	SELECT 4 id, 371 value
	UNION ALL
	SELECT 5 id, 193 value
	UNION ALL 
	SELECT 6 id, 1923 value
	UNION ALL
	SELECT 7 id, 1022 value
	UNION ALL
	SELECT 8 id, 162 vlaue
	UNION ALL 
	SELECT 9 id, 19234785 value
	UNION ALL
	SELECT 10 id, 5673 value
)
SELECT 
	*,
	FARM_FINGERPRINT(CAST(value AS string)) AS farm_fingerprint_value,
	ROW_NUMBER() OVER (
		ORDER BY FARM_FINGERPRINT(
			CAST(value AS string)
		)) AS random_count
FROM temp1
-- | id | value | farm_fingerprint_value | random_count |
-- | - | - | - | - |
-- | 4 | 371 | -5933768062887988196 | 1 |
-- | 7 | 1022 | -3328868577810523882 | 2 |
-- | 9 | 19234785 | -3020018155477927287 | 3 |
-- ...
-- | 10 | 5 | 193 | 8834274070702014843 | 10 |
```

Therefore, we can choose to randomly select a sample of 3 rows, but this will be reproducible in the future.
```sql
WITH temp1 AS (
	SELECT 1 id, 103 value
	UNION ALL
	SELECT 2 id, 102 value
	UNION ALL
	SELECT 3 id, 1339 value
	UNION ALL
	SELECT 4 id, 371 value
	UNION ALL
	SELECT 5 id, 193 value
	UNION ALL 
	SELECT 6 id, 1923 value
	UNION ALL
	SELECT 7 id, 1022 value
	UNION ALL
	SELECT 8 id, 162 vlaue
	UNION ALL 
	SELECT 9 id, 19234785 value
	UNION ALL
	SELECT 10 id, 5673 value
),
hashed AS (
	SELECT 
		*,
		FARM_FINGERPRINT(CAST(value AS string)) AS farm_fingerprint_value, -- NOTE : this is an optional variable, used here just for visualising
		ROW_NUMBER() OVER (
			ORDER BY FARM_FINGERPRINT(
				CAST(value AS string)
			)) AS random_count
	FROM temp1
)
SELECT 
	id,
	value
FROM hashed
WHERE random_count <= 3
-- | id | value |
-- | - | - |
-- | 4 | 371 |
-- | 7 | 1022 |
-- | 9 | 19234785 |
```

Using a slighly modified example to do stratified sampling stratified by groupp:
```sql
WITH temp1 AS (
	SELECT 1 id, 103 value, 'a' groupp
	UNION ALL
	SELECT 2 id, 102 value, 'a' groupp
	UNION ALL
	SELECT 3 id, 1339 value, 'a' groupp
	UNION ALL
	SELECT 4 id, 371 value, 'a' groupp
	UNION ALL
	SELECT 5 id, 193 value, 'b' groupp
	UNION ALL 
	SELECT 6 id, 1923 value, 'b' groupp
	UNION ALL
	SELECT 7 id, 1022 value, 'b' groupp
	UNION ALL
	SELECT 8 id, 162 vlaue, 'c' groupp
	UNION ALL 
	SELECT 9 id, 19234785 value, 'c' groupp
	UNION ALL
	SELECT 10 id, 5673 value, 'c' groupp
),
/*
CODE VARIANT 1
this is the way to write it in PostgreSQL; also works in BigQuery but is more verbose
*/
hashed AS (
	SELECT 
		*,
		ROW_NUMBER() OVER (
			PARTITION BY groupp
      ORDER BY FARM_FINGERPRINT(
				CAST(value AS string)
			)) AS random_count
	FROM temp1
)
SELECT 
	id,
  groupp,
	value
FROM hashed
WHERE random_count <= 2
/*
CODE VARIANT 2
In BigQuery, you can make it shorter:
*/
SELECT *
FROM temp1 
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY groupp 
  ORDER BY FARM_FINGERPRINT(CAST(value AS string))
) <= 3

-- | id | groupp | value |
-- | 4 | a | 371 |
-- |3 | a | 1339 |
-- | 7 | b | 1022 |
-- | 6 | b | 1923 |
-- | 9 | c | 19234785 |
-- | 10 | c | 5673 |
```

Another example:
```sql
with temp1 AS (
  select 1 idd, 'one' descr
  union all 
  select 1 idd, 'one again' descr
  union all
  select 2 idd, 'two' descr
  union all 
  select 2 idd, 'two again' descr 
  union all 
  select 3 idd, 'three' descr
  union all 
  select 4 idd, 'four' descr 
  union all 
  select 5 idd, 'five' descr
)

select 
  idd,
  string_agg(descr)
from temp1
GROUP BY idd 
ORDER BY farm_fingerprint(cast(idd as string))
limit 3
-- Fila	idd	f0_
-- 1	1	one,one again
-- 2	3	three
-- 3	4	four	
```

---

Final note:

apart from farm_fingerprint (which is great because it's reproducible) you can also use random number: 

```sql
SELECT *
FROM temp1 
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY groupp 
  ORDER BY RAND()
) <= 3
```


